# AKANE E14-2 v2：継ぎ足しの資料を Wikipedia の最初の1文に替える（取り方を直した版）

v1 では Wikipedia に「アクセスが多すぎ」と断られ、半分以上の語が取れませんでした。v2 は 1回で20語まとめて聞き、1本ずつ1秒あけ、断られたら待ちます。取れない語が5%を超えたら止まります。

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：plans/E14-2_wikipedia_first_sentence.md。

- E11-2 の6段の鎖をそのまま読み、鎖の全部の語に、**Wikipedia のその語の記事の最初の1文**を足します（Wikipedia の公開 API で記事の冒頭を取ります。曖昧さ回避のページは使いません）。
- 文の主語がその語の時だけ使い、文の中のほかの名詞から親を選びます。判定（「この文は X が Y の一種だと言っていますか？」）と親の語の意味選びは Gemma です（E14-1 と同じ）。
- 一般的な文（GenericsKB）・自作文は使いません。
- 正解はこのノートに入っていません。手で書いた語の一覧もありません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（今までと同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E14-2_results/E142v2_grow.json.gz` ができます。
5. 時間の見込み：40〜60分（Wikipedia は1段に数分、継ぎ足し30〜40分）。
   途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所（継ぎ足し）は飛ばします。

In [ ]:
import os, json, time, subprocess, sys, gzip, shutil, collections, re
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E14-2_results"
os.makedirs(OUT, exist_ok=True)
E112 = f"{BASE}/E11-2_results/E11-2_results.json"
WIKT = f"{BASE}/E10-1_results/E101_wiktionary.json.gz"   # E10-1 で読んだ名詞の辞書（新しく着く語に使う）
for p in (E112, WIKT):
    assert os.path.exists(p), "見つかりません: " + p
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E11と同じ
GEMMA_ID = "google/gemma-2-2b-it"
GKB_ID = "community-datasets/generics_kb"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)
def save_gz(obj, name):
    p = f"{OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name):
    p = f"{OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身1（analysis/nolist/e8n_core.py と同じ。手書きの語の一覧はありません）
"""E8の中身（Colabと手元の両方で使う）。
E7との違い：
- ダンプを読む時に、意味ごとの印（{{lb|en|slang}} など）を残す。途中の語の意味は8個まで持つ。
- 鎖の点を「語」ではなく「語＋意味の番号」にし、途中の語の意味を選ぶ方法を4通り：
  A 選ばない（E7と同じ、途中の語は最初の4つの意味を全部）
  B 印（古い・まれ・俗語など）のない最初の意味（Kovács 2022型）
  C 1つ前の説明文に一番近い意味（文の近さ、McIntosh 2009・OntoLearn型。分類Cは入れない）
  D Gemmaに「この説明文の中の語 g はどの意味か」を選ばせる（Meconi 2025・ConSeC型。分類Cは入れない）
- 各点から、46分類すべてへの「着き」をNLIで測る（打ち消し＝両立しない分類に着くか、に使う）。
"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, A_MID, DEPTH = 8, 8, 4, 3
# 一覧なし版（2026-11-11、本人「ハードコードは絶対ダメ」）：手で書いた語の一覧（印・頭の名詞の決まり）を使わない。
# 頭の名詞は文法（spaCyの名詞のかたまりと係り受け）だけで候補を出し、どれを使うかはNLIの段の値で決める（Colab側 set_genus）。


def genus_candidates(defn, nlp, max_c=4):
    """最初の名詞のかたまりの頭、その頭に付いた of の目的語（あれば）。それぞれ見出し形と、そのままの形（違う時）。名詞・固有名詞だけ。"""
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    doc = nlp(t[:200])
    chunks = [c for c in doc.noun_chunks if c.root.pos_ in ("NOUN", "PROPN")]
    if not chunks:
        return []
    root = chunks[0].root
    toks = [root] + [o for ch in root.children if ch.dep_ == "prep" and ch.text.lower() == "of"
                     for o in ch.children if o.dep_ == "pobj" and o.pos_ in ("NOUN", "PROPN")]
    out = []
    for tk in toks:
        for w in (tk.lemma_.lower(), tk.text.lower()):
            if w not in out and re.fullmatch(r"[a-z][a-z\-]*", w):
                out.append(w)
    return out[:max_c]


# --- ダンプの読み取り（E7と同じ書き換え。ただし lb の印は取っておく） ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)
LB = re.compile(r"\{\{\s*(?:lb|lbl|label|tlb)\s*\|\s*en\s*\|([^{}]*)\}\}")


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), [p for p in parts[1:] if "=" not in p]
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return args[1] if len(args) > 1 else ""
    if name in ARG1_T:
        return args[0] if args else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    return re.sub(r"\s+", " ", t).strip(" ;,.") + "."


def labels_of(raw):
    out = []
    for m in LB.finditer(raw):
        for p in m.group(1).split("|"):
            p = p.strip().lower()
            if p and "=" not in p and p not in ("_", "and", "or"):
                out.append(p)
    return out


def noun_senses_from_wikitext(wt, limit=MAX_DEFS_MID):
    """[[説明文, [印]], ...]（E7と同じ説明文の並び。印は {{lb|en|...}} から）"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*Noun\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if not line.startswith("# "):
                continue
            raw = line[2:].strip()
            if SKIP_T.match(raw):
                continue
            d = strip_wikitext(raw)
            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
                out.append([d[:220], labels_of(raw)])
    return out[:limit]


def parse_dump(fobj, log=print, every=500000):
    import xml.etree.ElementTree as ET
    out, n, title, ns = {}, 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and re.fullmatch(r"[a-z][a-z\-]*", title) and el.text and "==English==" in el.text:
                ds = noun_senses_from_wikitext(el.text)
                if ds:
                    out[title] = ds
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(out):,}")
    return out


# --- 鎖をたどる ---
def build_lexicon(items, WKD, nlp, log=print):
    """L[w] = [{"def", "labels", "genus"}]。物の語は items の wk_defs（印なし）か WKD、途中の語は WKD から8個まで。
    A用の段（途中の語は最初の4つ全部）と、B/C/D用の段（途中の語は8個から1つ選ぶ）の両方で出てくる語をすべて集める。"""
    L = {}
    def senses(w, ent_defs=None):
        if w in L:
            return L[w]
        if ent_defs is not None:
            ss = [[d, []] for d in ent_defs][:MAX_DEFS_ENT]
        else:
            ss = WKD.get(w, [])[:MAX_DEFS_MID]
        L[w] = [{"def": d, "labels": lb, "cands": genus_candidates(d, nlp), "genus": None} for d, lb in ss]
        return L[w]
    front = set()
    for it in items:
        ed = it.get("wk_defs")
        senses(it["ent"], ed if ed else None)
        front.add(it["ent"])
    for depth in range(DEPTH):
        nxt = set()
        for w in front:
            for x in L[w]:
                for g in x["cands"]:
                    if g not in L:
                        senses(g)
                        nxt.add(g)
        log(f"段{depth + 1}: 新しい語 {len(nxt)}  語の合計 {len(L)}")
        front = nxt
    return L


def set_genus(L, SC):
    """SC[(w,k,g)] = NLIの「含意」（"w" means: d ⇒ W is a type of g.）。候補のうち辞書にあり、一番値の大きい物を頭の名詞にする（一覧なし）。"""
    for w, xs in L.items():
        for k, x in enumerate(xs):
            cs = [g for g in x["cands"] if g != w and L.get(g)]
            x["genus"] = max(cs, key=lambda g: SC.get((w, k, g), 0.0)) if cs else None
    return L


def edges(L, ents):
    """(w,k) で genus g を持つ段のうち、たどり得るもの（物の語から3段まで、途中の語は全意味）"""
    seen, front, out = set(ents), set(ents), set()
    for _ in range(DEPTH):
        nxt = set()
        for w in front:
            for k, x in enumerate(L.get(w, [])):
                g = x["genus"]
                if g and g in L and g != w:
                    out.add((w, k))
                    if g not in seen:
                        seen.add(g); nxt.add(g)
        front = nxt
    return sorted(out)


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def premise(w, d):
    return f"\"{w}\" means: {d}"


def hyp(w, c):
    return f"{w[:1].upper() + w[1:]} is a type of {c}."




def chain_nodes(L, x, method, choice=None):
    """物の語 x から3段までの点（語, 意味の番号）と段（(w,k)→(g,j)）。
    A：途中の語は最初の4つの意味を全部（E7と同じ）。B/C/D：途中の語は choice[(w,k)] で1つに決める。物の語は全部の意味。"""
    nodes, links = set(), set()
    front = [(x, k) for k in range(len(L.get(x, [])))]
    nodes.update(front)
    for _ in range(DEPTH):
        nxt = []
        for (w, k) in front:
            g = L[w][k]["genus"]
            if not g or g not in L or g == w or not L[g]:
                continue
            if method == "A":
                js = range(min(A_MID, len(L[g])))
            else:
                js = [choice.get((w, k), 0) if len(L[g]) > 1 else 0]
            for j in js:
                links.add(((w, k), (g, j)))
                if (g, j) not in nodes:
                    nodes.add((g, j)); nxt.append((g, j))
        front = nxt
    return nodes, links

In [ ]:
# 中身2（analysis/e14/e14_core.py と同じ。手書きの語の一覧はありません）
"""E14 の中身：辞書の鎖（E11-2 の6段の鎖＝E8n 方法D を伸ばした物）の各点に、
別の資料の「A は B の一種」を継ぎ足す（計画 plans/E14_fill_gaps.md）。手書きの語の一覧はない。
e8n_core（genus_candidates, choice_prompt, premise, hyp, MAX_DEFS_MID）の後に置いて使う。
NLI・Gemma・文の資料は呼び出しの関数として外から渡す（Colab ではモデル、手元の確かめでは作り物）。

点の種類：
  ("s", w, k)  辞書の意味（E8n と同じ。L[w][k]）
  ("g", w, i)  継ぎ足しの文（GR[w][i]。src は "g"＝一般的な文、"s"＝Gemma が書いた文）
つながり：
  ("s",w,k) → ("s",g,j)   辞書の頭の語（強さ ST[(w,k)]、1段）
  ("s",w,k) → ("g",w,i)   同じ語の文。文の中の w の意味が k の時だけ（Gemma が選ぶ）。物の語の文は根から直接。強さ1、段に数えない
  ("g",w,i) → ("s",p,j)   文の中の親の語 p（強さ＝NLI「文 ⇒ W is a type of p.」、1段）。p の意味 j は Gemma が選ぶ
"""
import re

MAX_DEPTH = 6        # E11-2 と同じ上限
K_GKB = 3            # 一般的な文は1語につき3文まで（親への NLI の値が高い順）
N_GKB_READ = 20      # 1語につき読む一般的な文（資料の score の高い順）
MAX_CANDS = 6        # 1文の親の候補の数
SELF_TOKENS = 40     # 書かせる長さ（30語ほど）
WORD = re.compile(r"[a-z][a-z\-]*")


def self_prompt(w):
    return f"Write one short sentence saying what kind of thing a \"{w}\" is."


def sense_in_sentence_prompt(sent, w, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Sentence: {sent}\n\nIn this sentence, which meaning of the word \"{w}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def clean_self(text):
    """書かせた文：最初の空でない行。行の頭の記号を外す（書式の読み方）。"""
    for line in text.splitlines():
        s = re.sub(r"^[\s\-\*\d\.\)\"]+", "", line).strip()
        if len(s.split()) >= 3:
            return s
    return None


def subject_parents(doc, w):
    """文の主語（名詞のかたまりで、係りが主語）の頭が w の時、文の中のほかの名詞のかたまりの頭（見出し形）を親の候補にする。
    w が主語でなければ None。どの候補を親にするかは NLI で決める（ここでは決めない）。"""
    chunks = list(doc.noun_chunks)
    subj = [c for c in chunks if c.root.dep_ in ("nsubj", "nsubjpass")
            and w in (c.root.lemma_.lower(), c.root.text.lower())]
    if not subj:
        return None
    s = subj[0]
    out = []
    for c in chunks:
        if c.start == s.start:
            continue
        if c.root.pos_ != "NOUN":
            continue
        lw = c.root.lemma_.lower()
        if lw != w and WORD.fullmatch(lw) and lw not in out:
            out.append(lw)
    return out[:MAX_CANDS]


def new_word(L, w, WKD, nlp):
    """辞書にまだ無い語を WKD（E10-1 の名詞の辞書）から読む。頭の名詞はまだ決めない。"""
    if w in L:
        return bool(L[w])
    L[w] = [{"def": d, "labels": lb, "cands": genus_candidates(d, nlp), "genus": None}
            for d, lb in WKD.get(w, [])[:MAX_DEFS_MID]]
    return bool(L[w])


def follow(L, choice, GR, x, maxd, srcs=("g", "s")):
    """物の語 x から maxd 段まで。点 → 一番浅い段の数、つながりの組。srcs で使う資料を選ぶ（() なら E11-2 の鎖と同じ）。"""
    depth, links = {}, set()
    front = []
    for k in range(len(L.get(x, []))):
        depth[("s", x, k)] = 0; front.append(("s", x, k))
    for i, g in enumerate(GR.get(x, [])):
        if g["src"] in srcs and g.get("parent"):
            depth[("g", x, i)] = 0; front.append(("g", x, i))
    d = 0
    while front and d < maxd:
        nxt = []
        for n in front:
            outs = []
            if n[0] == "s":
                _, w, k = n
                gw = L[w][k].get("genus")
                if gw and gw in L and gw != w and L[gw]:
                    j = choice.get((w, k), 0) if len(L[gw]) > 1 else 0
                    outs.append((("s", gw, j), 1))
                if w != x:   # 途中の語の文（意味が合う時だけ、段に数えない）
                    for i, g in enumerate(GR.get(w, [])):
                        if g["src"] in srcs and g.get("parent") and g.get("wsense", 0) == k:
                            outs.append((("g", w, i), 0))
            else:
                _, w, i = n
                g = GR[w][i]
                outs.append((("s", g["parent"], g.get("psense", 0)), 1))
            for m, step in outs:
                links.add((n, m))
                dm = depth[n] + step
                if m not in depth or dm < depth[m]:
                    depth[m] = dm
                    if step == 0:
                        front.append(m) if dm == d else nxt.append(m)
                    else:
                        nxt.append(m)
        front = nxt
        d += 1
    return depth, links


def grow(items, L, choice, ST, AR, GR, ARG, WKD, nlp, nli, nli_sent, gemma_choice, gemma_sense, gkb_sents, self_write,
         maxd=MAX_DEPTH, log=print, wiki_sents=None):
    """1段ずつ広げる。（E14-2：wiki_sents([語])→{語: 文 or None} を渡すと、Wikipedia の最初の1文を src "w" として足す）各段で、先の点の語について：
    a. 文を集める（一般的な文 N_GKB_READ 文・書かせた文1文）→ 主語が w の文だけ、親の候補を出す
    b. 親の候補を NLI（文 ⇒ W is a type of 候補）で測り、一番の候補を親に。一般的な文は値の高い順に K_GKB 文
    c. 親の語を辞書から読み、親の意味・文の中の w の意味を Gemma が選ぶ
    d. 辞書の意味の点は E11-2 と同じく 頭の名詞・段の値・意味選び（新しい語だけ）
    e. 新しく着いた点の、問題の分類への着き（辞書の意味は E8n と同じ NLI、文は 文 ⇒ W is a type of C.）
    関数：nli([(w,k,c)])→[(含意,矛盾)]（"w" means: 説明文 ⇒ W is a type of c.）、nli_sent([(文, w, c)])→[(含意,矛盾)]、
    gemma_choice([(w,k)])→[番号]（E8n の方法D）、gemma_sense([(文, 語, [説明文])])→[番号]、
    gkb_sents([語])→{語: [文,...]}（score の高い順）、self_write([語])→{語: 文 or None}。"""
    stat = {"words_grafted": 0, "gkb_sent_read": 0, "gkb_kept": 0, "self_kept": 0, "new_words": 0, "arrive": 0}
    done = set(GR)
    n0 = len(L)
    for lev in range(maxd):
        fr = set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev)
            fr |= {n for n, d in dep.items() if d == lev}
        words = sorted({n[1] for n in fr if n[0] == "s"} - done)
        # a. 文を集める
        G = gkb_sents(words) if words else {}
        S = self_write(words) if words else {}
        Wk = wiki_sents(words) if (words and wiki_sents) else {}
        cand = []   # (w, src, 文, [候補])
        for w in words:
            for s in G.get(w, [])[:N_GKB_READ]:
                ps = subject_parents(nlp(s), w)
                if ps:
                    cand.append((w, "g", s, ps))
            stat["gkb_sent_read"] += len(G.get(w, [])[:N_GKB_READ])
            for src, s in (("s", S.get(w)), ("w", Wk.get(w))):
                if s:
                    ps = subject_parents(nlp(s), w)
                    if ps:
                        cand.append((w, src, s, ps))
        # 親の候補は辞書にある語だけ
        cand = [(w, src, s, [p for p in ps if new_word(L, p, WKD, nlp)]) for w, src, s, ps in cand]
        cand = [c for c in cand if c[3]]
        # b. 親を NLI で決める
        trip = sorted({(s, w, p) for w, src, s, ps in cand for p in ps})
        sc = dict(zip(trip, nli_sent(trip))) if trip else {}
        per = {}
        for w, src, s, ps in cand:
            best = max(ps, key=lambda p: sc[(s, w, p)][0])
            per.setdefault((w, src), []).append({"src": src, "sent": s, "parent": best, "step": sc[(s, w, best)],
                                                 "cands": ps})
        for w in words:
            gs = sorted(per.get((w, "g"), []), key=lambda g: -g["step"][0])[:K_GKB]
            ss = per.get((w, "s"), [])[:1]
            ws = per.get((w, "w"), [])[:1]
            GR[w] = gs + ss + ws
            done.add(w)
            stat["words_grafted"] += 1; stat["gkb_kept"] += len(gs); stat["self_kept"] += len(ss)
        # c. 親の意味と、文の中の w の意味（意味が2つ以上の時だけ Gemma）
        jobs, keys = [], []
        for w in words:
            for i, g in enumerate(GR[w]):
                p = g["parent"]
                if len(L[p]) > 1:
                    jobs.append((g["sent"], p, [x["def"] for x in L[p]])); keys.append((w, i, "psense"))
                else:
                    g["psense"] = 0
                if len(L.get(w, [])) > 1:
                    jobs.append((g["sent"], w, [x["def"] for x in L[w]])); keys.append((w, i, "wsense"))
                else:
                    g["wsense"] = 0
        if jobs:
            for (w, i, f), v in zip(keys, gemma_sense(jobs)):
                GR[w][i][f] = int(v)
        # d. 辞書の意味の点：新しい語の頭の名詞・段の値・意味選び（E11-2 の extend と同じ手順）
        dep_all = set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev + 1)
            dep_all |= set(dep)
        sn = sorted(n for n in dep_all if n[0] == "s" and L[n[1]][n[2]].get("genus") is None and L[n[1]][n[2]]["cands"]
                    and (n[1], n[2]) not in ST)
        for _, w, k in sn:
            for c in L[w][k]["cands"]:
                if c != w and c not in L:
                    stat["new_words"] += int(new_word(L, c, WKD, nlp))
        pairs = sorted({(w, k, c) for _, w, k in sn for c in L[w][k]["cands"] if c != w and L.get(c)})
        gsc = dict(zip(pairs, nli(pairs))) if pairs else {}
        for _, w, k in sn:
            cs = [c for c in L[w][k]["cands"] if (w, k, c) in gsc]
            if cs:
                L[w][k]["genus"] = max(cs, key=lambda c: gsc[(w, k, c)][0])
                ST[(w, k)] = gsc[(w, k, L[w][k]["genus"])]
        cj = sorted((n[1], n[2]) for n in dep_all if n[0] == "s" and L[n[1]][n[2]].get("genus")
                    and len(L.get(L[n[1]][n[2]]["genus"], [])) > 1 and (n[1], n[2]) not in choice)
        if cj:
            for n, v in zip(cj, gemma_choice(cj)):
                choice[n] = int(v)
        # e. 着き
        dep_all = {}
        arr_s, arr_g = set(), set()
        for it in items:
            dep, _ = follow(L, choice, GR, it["ent"], lev + 1)
            C = it["catw"]
            for n in dep:
                if n[0] == "s" and (n[1], n[2], C) not in AR:
                    arr_s.add((n[1], n[2], C))
                elif n[0] == "g" and (n[1], n[2], C) not in ARG:
                    arr_g.add((n[1], n[2], C))
        arr_s, arr_g = sorted(arr_s), sorted(arr_g)
        if arr_s:
            for n, v in zip(arr_s, nli(arr_s)):
                AR[n] = v
        if arr_g:
            for n, v in zip(arr_g, nli_sent([(GR[w][i]["sent"], w, C) for w, i, C in arr_g])):
                ARG[n] = v
        stat["arrive"] += len(arr_s) + len(arr_g)
        log(f"段{lev + 1}: 先の点 {len(fr)}  文を探した語 {len(words)}  文の候補 {len(cand)}  新しい語 {stat['new_words']}  "
            f"着き {len(arr_s)}+{len(arr_g)}")
    stat["L_words_added"] = len(L) - n0
    return stat


def chains(items, L, choice, GR, maxd=MAX_DEPTH):
    """問題ごとの点（種類, 語, 番号, 段の数）とつながり。分析は資料（g・s）を選んで鎖を作り直す。"""
    out = {}
    for it in items:
        dep, links = follow(L, choice, GR, it["ent"], maxd)
        out[it["id"]] = ([[t, w, k, d] for (t, w, k), d in sorted(dep.items())],
                         sorted([list(a), list(b)] for a, b in links))
    return out

In [ ]:
OLD = json.loads(r'''[{"id": 0, "ent": "barbell", "catw": "tool", "stmt": "Barbell is a type of tool."}, {"id": 1, "ent": "tenement", "catw": "building", "stmt": "Tenement is a type of building."}, {"id": 2, "ent": "porpoise", "catw": "dog", "stmt": "Porpoise is a type of dog."}, {"id": 3, "ent": "juvenile", "catw": "boat", "stmt": "Juvenile is a type of boat."}, {"id": 4, "ent": "barbell", "catw": "mammal", "stmt": "Barbell is a type of mammal."}, {"id": 5, "ent": "grocer", "catw": "living thing", "stmt": "Grocer is a type of living thing."}, {"id": 6, "ent": "gramophone", "catw": "game", "stmt": "Gramophone is a type of game."}, {"id": 7, "ent": "hooch", "catw": "artist", "stmt": "Hooch is a type of artist."}, {"id": 8, "ent": "bilingual", "catw": "person", "stmt": "Bilingual is a type of person."}, {"id": 9, "ent": "minnow", "catw": "disease", "stmt": "Minnow is a type of disease."}, {"id": 10, "ent": "gazelle", "catw": "food", "stmt": "Gazelle is a type of food."}, {"id": 11, "ent": "gasket", "catw": "game", "stmt": "Gasket is a type of game."}, {"id": 12, "ent": "isolationist", "catw": "living thing", "stmt": "Isolationist is a type of living thing."}, {"id": 13, "ent": "clamp", "catw": "tool", "stmt": "Clamp is a type of tool."}, {"id": 14, "ent": "copycat", "catw": "body part", "stmt": "Copycat is a type of body part."}, {"id": 15, "ent": "peshmerga", "catw": "living thing", "stmt": "Peshmerga is a type of living thing."}, {"id": 16, "ent": "tabor", "catw": "boat", "stmt": "Tabor is a type of boat."}, {"id": 17, "ent": "stepmother", "catw": "person", "stmt": "Stepmother is a type of person."}, {"id": 18, "ent": "terrier", "catw": "bird", "stmt": "Terrier is a type of bird."}, {"id": 19, "ent": "preteen", "catw": "metal", "stmt": "Preteen is a type of metal."}, {"id": 20, "ent": "composer", "catw": "person", "stmt": "Composer is a type of person."}, {"id": 21, "ent": "suitor", "catw": "living thing", "stmt": "Suitor is a type of living thing."}, {"id": 22, "ent": "leaker", "catw": "person", "stmt": "Leaker is a type of person."}, {"id": 23, "ent": "actuator", "catw": "machine", "stmt": "Actuator is a type of machine."}, {"id": 24, "ent": "lally", "catw": "tool", "stmt": "Lally is a type of tool."}, {"id": 25, "ent": "epileptic", "catw": "disease", "stmt": "Epileptic is a type of disease."}, {"id": 26, "ent": "pew", "catw": "man-made object", "stmt": "Pew is a type of man-made object."}, {"id": 27, "ent": "microphone", "catw": "equipment", "stmt": "Microphone is a type of equipment."}, {"id": 28, "ent": "felt", "catw": "material", "stmt": "Felt is a type of material."}, {"id": 29, "ent": "primrose", "catw": "living thing", "stmt": "Primrose is a type of living thing."}, {"id": 30, "ent": "quartermaster", "catw": "person", "stmt": "Quartermaster is a type of person."}, {"id": 31, "ent": "delegate", "catw": "person", "stmt": "Delegate is a type of person."}, {"id": 32, "ent": "grubby", "catw": "toy", "stmt": "Grubby is a type of toy."}, {"id": 33, "ent": "prince", "catw": "artist", "stmt": "Prince is a type of artist."}, {"id": 34, "ent": "locomotive", "catw": "man-made object", "stmt": "Locomotive is a type of man-made object."}, {"id": 35, "ent": "lobbyist", "catw": "living thing", "stmt": "Lobbyist is a type of living thing."}, {"id": 36, "ent": "workman", "catw": "footwear", "stmt": "Workman is a type of footwear."}, {"id": 37, "ent": "stupid", "catw": "person", "stmt": "Stupid is a type of person."}, {"id": 38, "ent": "watchman", "catw": "body part", "stmt": "Watchman is a type of body part."}, {"id": 39, "ent": "headliner", "catw": "equipment", "stmt": "Headliner is a type of equipment."}, {"id": 40, "ent": "geranium", "catw": "living thing", "stmt": "Geranium is a type of living thing."}, {"id": 41, "ent": "guerrilla", "catw": "weapon", "stmt": "Guerrilla is a type of weapon."}, {"id": 42, "ent": "sheeting", "catw": "food", "stmt": "Sheeting is a type of food."}, {"id": 43, "ent": "consulate", "catw": "device", "stmt": "Consulate is a type of device."}, {"id": 44, "ent": "dinky", "catw": "artist", "stmt": "Dinky is a type of artist."}, {"id": 45, "ent": "caterer", "catw": "sport", "stmt": "Caterer is a type of sport."}, {"id": 46, "ent": "rickey", "catw": "tree", "stmt": "Rickey is a type of tree."}, {"id": 47, "ent": "headmistress", "catw": "worker", "stmt": "Headmistress is a type of worker."}, {"id": 48, "ent": "educator", "catw": "worker", "stmt": "Educator is a type of worker."}, {"id": 49, "ent": "golf", "catw": "metal", "stmt": "Golf is a type of metal."}, {"id": 50, "ent": "velvet", "catw": "man-made object", "stmt": "Velvet is a type of man-made object."}, {"id": 51, "ent": "biographer", "catw": "disease", "stmt": "Biographer is a type of disease."}, {"id": 52, "ent": "zoologist", "catw": "living thing", "stmt": "Zoologist is a type of living thing."}, {"id": 53, "ent": "doge", "catw": "living thing", "stmt": "Doge is a type of living thing."}, {"id": 54, "ent": "mimic", "catw": "reptile", "stmt": "Mimic is a type of reptile."}, {"id": 55, "ent": "passerby", "catw": "organ", "stmt": "Passerby is a type of organ."}, {"id": 56, "ent": "dieter", "catw": "fish", "stmt": "Dieter is a type of fish."}, {"id": 57, "ent": "niqab", "catw": "reptile", "stmt": "Niqab is a type of reptile."}, {"id": 58, "ent": "bulldozer", "catw": "machine", "stmt": "Bulldozer is a type of machine."}, {"id": 59, "ent": "deflector", "catw": "equipment", "stmt": "Deflector is a type of equipment."}, {"id": 60, "ent": "schoolgirl", "catw": "metal", "stmt": "Schoolgirl is a type of metal."}, {"id": 61, "ent": "falconer", "catw": "sport", "stmt": "Falconer is a type of sport."}, {"id": 62, "ent": "marlin", "catw": "drink", "stmt": "Marlin is a type of drink."}, {"id": 63, "ent": "plumber", "catw": "bird", "stmt": "Plumber is a type of bird."}, {"id": 64, "ent": "sarcophagus", "catw": "man-made object", "stmt": "Sarcophagus is a type of man-made object."}, {"id": 65, "ent": "amputee", "catw": "person", "stmt": "Amputee is a type of person."}, {"id": 66, "ent": "kimono", "catw": "fish", "stmt": "Kimono is a type of fish."}, {"id": 67, "ent": "temp", "catw": "worker", "stmt": "Temp is a type of worker."}, {"id": 68, "ent": "propeller", "catw": "reptile", "stmt": "Propeller is a type of reptile."}, {"id": 69, "ent": "wombat", "catw": "building", "stmt": "Wombat is a type of building."}, {"id": 70, "ent": "paintbrush", "catw": "tool", "stmt": "Paintbrush is a type of tool."}, {"id": 71, "ent": "womanizer", "catw": "body part", "stmt": "Womanizer is a type of body part."}, {"id": 72, "ent": "greenhouse", "catw": "man-made object", "stmt": "Greenhouse is a type of man-made object."}, {"id": 73, "ent": "tactician", "catw": "food", "stmt": "Tactician is a type of food."}, {"id": 74, "ent": "achiever", "catw": "fruit", "stmt": "Achiever is a type of fruit."}, {"id": 75, "ent": "moll", "catw": "person", "stmt": "Moll is a type of person."}, {"id": 76, "ent": "bonsai", "catw": "plant", "stmt": "Bonsai is a type of plant."}, {"id": 77, "ent": "underdog", "catw": "tree", "stmt": "Underdog is a type of tree."}, {"id": 78, "ent": "loom", "catw": "car", "stmt": "Loom is a type of car."}, {"id": 79, "ent": "individualist", "catw": "person", "stmt": "Individualist is a type of person."}, {"id": 80, "ent": "easel", "catw": "sport", "stmt": "Easel is a type of sport."}, {"id": 81, "ent": "finalist", "catw": "athlete", "stmt": "Finalist is a type of athlete."}, {"id": 82, "ent": "sagebrush", "catw": "weapon", "stmt": "Sagebrush is a type of weapon."}, {"id": 83, "ent": "uniform", "catw": "equipment", "stmt": "Uniform is a type of equipment."}, {"id": 84, "ent": "bonsai", "catw": "animal", "stmt": "Bonsai is a type of animal."}, {"id": 85, "ent": "receptionist", "catw": "worker", "stmt": "Receptionist is a type of worker."}, {"id": 86, "ent": "commoner", "catw": "worker", "stmt": "Commoner is a type of worker."}, {"id": 87, "ent": "modernist", "catw": "person", "stmt": "Modernist is a type of person."}, {"id": 88, "ent": "cuppa", "catw": "container", "stmt": "Cuppa is a type of container."}, {"id": 89, "ent": "cornerback", "catw": "athlete", "stmt": "Cornerback is a type of athlete."}, {"id": 90, "ent": "minnow", "catw": "game", "stmt": "Minnow is a type of game."}, {"id": 91, "ent": "neurologist", "catw": "worker", "stmt": "Neurologist is a type of worker."}, {"id": 92, "ent": "kitten", "catw": "mammal", "stmt": "Kitten is a type of mammal."}, {"id": 93, "ent": "undercarriage", "catw": "device", "stmt": "Undercarriage is a type of device."}, {"id": 94, "ent": "reporter", "catw": "worker", "stmt": "Reporter is a type of worker."}, {"id": 95, "ent": "critter", "catw": "equipment", "stmt": "Critter is a type of equipment."}, {"id": 96, "ent": "granddaughter", "catw": "mammal", "stmt": "Granddaughter is a type of mammal."}, {"id": 97, "ent": "crusher", "catw": "machine", "stmt": "Crusher is a type of machine."}, {"id": 98, "ent": "accomplice", "catw": "athlete", "stmt": "Accomplice is a type of athlete."}, {"id": 99, "ent": "dubbing", "catw": "device", "stmt": "Dubbing is a type of device."}, {"id": 100, "ent": "tortoise", "catw": "reptile", "stmt": "Tortoise is a type of reptile."}, {"id": 101, "ent": "jukebox", "catw": "equipment", "stmt": "Jukebox is a type of equipment."}, {"id": 102, "ent": "restaurant", "catw": "building", "stmt": "Restaurant is a type of building."}, {"id": 103, "ent": "sectional", "catw": "man-made object", "stmt": "Sectional is a type of man-made object."}, {"id": 104, "ent": "heifer", "catw": "living thing", "stmt": "Heifer is a type of living thing."}, {"id": 105, "ent": "kea", "catw": "living thing", "stmt": "Kea is a type of living thing."}, {"id": 106, "ent": "minion", "catw": "living thing", "stmt": "Minion is a type of living thing."}, {"id": 107, "ent": "opportunist", "catw": "person", "stmt": "Opportunist is a type of person."}, {"id": 108, "ent": "conformist", "catw": "artist", "stmt": "Conformist is a type of artist."}, {"id": 109, "ent": "burglar", "catw": "body part", "stmt": "Burglar is a type of body part."}, {"id": 110, "ent": "buddy", "catw": "man-made object", "stmt": "Buddy is a type of man-made object."}, {"id": 111, "ent": "wraparound", "catw": "metal", "stmt": "Wraparound is a type of metal."}, {"id": 112, "ent": "vise", "catw": "tool", "stmt": "Vise is a type of tool."}, {"id": 113, "ent": "helicopter", "catw": "man-made object", "stmt": "Helicopter is a type of man-made object."}, {"id": 114, "ent": "calender", "catw": "tool", "stmt": "Calender is a type of tool."}, {"id": 115, "ent": "snob", "catw": "game", "stmt": "Snob is a type of game."}, {"id": 116, "ent": "drunkard", "catw": "living thing", "stmt": "Drunkard is a type of living thing."}, {"id": 117, "ent": "bandicoot", "catw": "disease", "stmt": "Bandicoot is a type of disease."}, {"id": 118, "ent": "musket", "catw": "vehicle", "stmt": "Musket is a type of vehicle."}, {"id": 119, "ent": "grader", "catw": "sport", "stmt": "Grader is a type of sport."}, {"id": 120, "ent": "briefcase", "catw": "vegetable", "stmt": "Briefcase is a type of vegetable."}, {"id": 121, "ent": "alternator", "catw": "vehicle", "stmt": "Alternator is a type of vehicle."}, {"id": 122, "ent": "wallaby", "catw": "plant", "stmt": "Wallaby is a type of plant."}, {"id": 123, "ent": "interceptor", "catw": "aircraft", "stmt": "Interceptor is a type of aircraft."}, {"id": 124, "ent": "carbine", "catw": "equipment", "stmt": "Carbine is a type of equipment."}, {"id": 125, "ent": "spitfire", "catw": "living thing", "stmt": "Spitfire is a type of living thing."}, {"id": 126, "ent": "huntress", "catw": "worker", "stmt": "Huntress is a type of worker."}, {"id": 127, "ent": "heathen", "catw": "flower", "stmt": "Heathen is a type of flower."}, {"id": 128, "ent": "senna", "catw": "man-made object", "stmt": "Senna is a type of man-made object."}, {"id": 129, "ent": "macho", "catw": "animal", "stmt": "Macho is a type of animal."}, {"id": 130, "ent": "beloved", "catw": "insect", "stmt": "Beloved is a type of insect."}, {"id": 131, "ent": "wallaby", "catw": "living thing", "stmt": "Wallaby is a type of living thing."}, {"id": 132, "ent": "accountant", "catw": "game", "stmt": "Accountant is a type of game."}, {"id": 133, "ent": "shogun", "catw": "fabric", "stmt": "Shogun is a type of fabric."}, {"id": 134, "ent": "parsonage", "catw": "boat", "stmt": "Parsonage is a type of boat."}, {"id": 135, "ent": "timepiece", "catw": "tool", "stmt": "Timepiece is a type of tool."}, {"id": 136, "ent": "biter", "catw": "living thing", "stmt": "Biter is a type of living thing."}, {"id": 137, "ent": "rugby", "catw": "equipment", "stmt": "Rugby is a type of equipment."}, {"id": 138, "ent": "bedroom", "catw": "building", "stmt": "Bedroom is a type of building."}, {"id": 139, "ent": "wombat", "catw": "tree", "stmt": "Wombat is a type of tree."}, {"id": 140, "ent": "everlasting", "catw": "game", "stmt": "Everlasting is a type of game."}, {"id": 141, "ent": "auctioneer", "catw": "material", "stmt": "Auctioneer is a type of material."}, {"id": 142, "ent": "theorist", "catw": "person", "stmt": "Theorist is a type of person."}, {"id": 143, "ent": "skater", "catw": "person", "stmt": "Skater is a type of person."}, {"id": 144, "ent": "mariner", "catw": "metal", "stmt": "Mariner is a type of metal."}, {"id": 145, "ent": "freelancer", "catw": "fish", "stmt": "Freelancer is a type of fish."}, {"id": 146, "ent": "dubbing", "catw": "game", "stmt": "Dubbing is a type of game."}, {"id": 147, "ent": "simulator", "catw": "game", "stmt": "Simulator is a type of game."}, {"id": 148, "ent": "circuitry", "catw": "sport", "stmt": "Circuitry is a type of sport."}, {"id": 149, "ent": "galleon", "catw": "boat", "stmt": "Galleon is a type of boat."}, {"id": 150, "ent": "steed", "catw": "animal", "stmt": "Steed is a type of animal."}, {"id": 151, "ent": "mosquito", "catw": "weapon", "stmt": "Mosquito is a type of weapon."}, {"id": 152, "ent": "legate", "catw": "game", "stmt": "Legate is a type of game."}, {"id": 153, "ent": "maxi", "catw": "room", "stmt": "Maxi is a type of room."}, {"id": 154, "ent": "bodybuilder", "catw": "animal", "stmt": "Bodybuilder is a type of animal."}, {"id": 155, "ent": "newlywed", "catw": "living thing", "stmt": "Newlywed is a type of living thing."}, {"id": 156, "ent": "finch", "catw": "material", "stmt": "Finch is a type of material."}, {"id": 157, "ent": "flea", "catw": "vegetable", "stmt": "Flea is a type of vegetable."}, {"id": 158, "ent": "betel", "catw": "tree", "stmt": "Betel is a type of tree."}, {"id": 159, "ent": "microphone", "catw": "sport", "stmt": "Microphone is a type of sport."}, {"id": 160, "ent": "airliner", "catw": "boat", "stmt": "Airliner is a type of boat."}, {"id": 161, "ent": "commodore", "catw": "machine", "stmt": "Commodore is a type of machine."}, {"id": 162, "ent": "billionaire", "catw": "body part", "stmt": "Billionaire is a type of body part."}, {"id": 163, "ent": "roadhouse", "catw": "game", "stmt": "Roadhouse is a type of game."}, {"id": 164, "ent": "hater", "catw": "person", "stmt": "Hater is a type of person."}, {"id": 165, "ent": "marc", "catw": "fruit", "stmt": "Marc is a type of fruit."}, {"id": 166, "ent": "tugboat", "catw": "disease", "stmt": "Tugboat is a type of disease."}, {"id": 167, "ent": "blouse", "catw": "fish", "stmt": "Blouse is a type of fish."}, {"id": 168, "ent": "houseboat", "catw": "device", "stmt": "Houseboat is a type of device."}, {"id": 169, "ent": "bovine", "catw": "toy", "stmt": "Bovine is a type of toy."}, {"id": 170, "ent": "porpoise", "catw": "athlete", "stmt": "Porpoise is a type of athlete."}, {"id": 171, "ent": "everlasting", "catw": "plant", "stmt": "Everlasting is a type of plant."}, {"id": 172, "ent": "tomahawk", "catw": "man-made object", "stmt": "Tomahawk is a type of man-made object."}, {"id": 173, "ent": "national", "catw": "boat", "stmt": "National is a type of boat."}, {"id": 174, "ent": "underpants", "catw": "man-made object", "stmt": "Underpants is a type of man-made object."}, {"id": 175, "ent": "hitchhiker", "catw": "worker", "stmt": "Hitchhiker is a type of worker."}, {"id": 176, "ent": "cutthroat", "catw": "plant", "stmt": "Cutthroat is a type of plant."}, {"id": 177, "ent": "theologian", "catw": "reptile", "stmt": "Theologian is a type of reptile."}, {"id": 178, "ent": "facilitator", "catw": "vegetable", "stmt": "Facilitator is a type of vegetable."}, {"id": 179, "ent": "rickey", "catw": "man-made object", "stmt": "Rickey is a type of man-made object."}, {"id": 180, "ent": "songwriter", "catw": "dish", "stmt": "Songwriter is a type of dish."}, {"id": 181, "ent": "extremist", "catw": "fruit", "stmt": "Extremist is a type of fruit."}, {"id": 182, "ent": "transvestite", "catw": "sport", "stmt": "Transvestite is a type of sport."}, {"id": 183, "ent": "warship", "catw": "fruit", "stmt": "Warship is a type of fruit."}, {"id": 184, "ent": "spokeswoman", "catw": "body part", "stmt": "Spokeswoman is a type of body part."}, {"id": 185, "ent": "brill", "catw": "animal", "stmt": "Brill is a type of animal."}, {"id": 186, "ent": "secessionist", "catw": "metal", "stmt": "Secessionist is a type of metal."}, {"id": 187, "ent": "beloved", "catw": "fabric", "stmt": "Beloved is a type of fabric."}, {"id": 188, "ent": "impala", "catw": "sport", "stmt": "Impala is a type of sport."}, {"id": 189, "ent": "firearm", "catw": "man-made object", "stmt": "Firearm is a type of man-made object."}, {"id": 190, "ent": "mathematician", "catw": "worker", "stmt": "Mathematician is a type of worker."}, {"id": 191, "ent": "buttercup", "catw": "man-made object", "stmt": "Buttercup is a type of man-made object."}, {"id": 192, "ent": "notary", "catw": "game", "stmt": "Notary is a type of game."}, {"id": 193, "ent": "missus", "catw": "boat", "stmt": "Missus is a type of boat."}, {"id": 194, "ent": "freelancer", "catw": "living thing", "stmt": "Freelancer is a type of living thing."}, {"id": 195, "ent": "beret", "catw": "toy", "stmt": "Beret is a type of toy."}, {"id": 196, "ent": "georgette", "catw": "clothing", "stmt": "Georgette is a type of clothing."}, {"id": 197, "ent": "parsonage", "catw": "man-made object", "stmt": "Parsonage is a type of man-made object."}, {"id": 198, "ent": "dongle", "catw": "metal", "stmt": "Dongle is a type of metal."}, {"id": 199, "ent": "spacecraft", "catw": "vehicle", "stmt": "Spacecraft is a type of vehicle."}, {"id": 200, "ent": "eunuch", "catw": "living thing", "stmt": "Eunuch is a type of living thing."}, {"id": 201, "ent": "apparel", "catw": "clothing", "stmt": "Apparel is a type of clothing."}, {"id": 202, "ent": "messenger", "catw": "person", "stmt": "Messenger is a type of person."}, {"id": 203, "ent": "strainer", "catw": "game", "stmt": "Strainer is a type of game."}, {"id": 204, "ent": "bookkeeper", "catw": "living thing", "stmt": "Bookkeeper is a type of living thing."}, {"id": 205, "ent": "clamp", "catw": "musical instrument", "stmt": "Clamp is a type of musical instrument."}, {"id": 206, "ent": "sieve", "catw": "body part", "stmt": "Sieve is a type of body part."}, {"id": 207, "ent": "madame", "catw": "insect", "stmt": "Madame is a type of insect."}, {"id": 208, "ent": "bethel", "catw": "body part", "stmt": "Bethel is a type of body part."}, {"id": 209, "ent": "moray", "catw": "disease", "stmt": "Moray is a type of disease."}, {"id": 210, "ent": "lemur", "catw": "animal", "stmt": "Lemur is a type of animal."}, {"id": 211, "ent": "orangutan", "catw": "fruit", "stmt": "Orangutan is a type of fruit."}, {"id": 212, "ent": "athlete", "catw": "living thing", "stmt": "Athlete is a type of living thing."}, {"id": 213, "ent": "vole", "catw": "worker", "stmt": "Vole is a type of worker."}, {"id": 214, "ent": "peony", "catw": "car", "stmt": "Peony is a type of car."}, {"id": 215, "ent": "undercarriage", "catw": "man-made object", "stmt": "Undercarriage is a type of man-made object."}, {"id": 216, "ent": "blacksmith", "catw": "artist", "stmt": "Blacksmith is a type of artist."}, {"id": 217, "ent": "erica", "catw": "game", "stmt": "Erica is a type of game."}, {"id": 218, "ent": "clinician", "catw": "person", "stmt": "Clinician is a type of person."}, {"id": 219, "ent": "beekeeper", "catw": "person", "stmt": "Beekeeper is a type of person."}, {"id": 220, "ent": "purveyor", "catw": "living thing", "stmt": "Purveyor is a type of living thing."}, {"id": 221, "ent": "carbine", "catw": "man-made object", "stmt": "Carbine is a type of man-made object."}, {"id": 222, "ent": "ani", "catw": "animal", "stmt": "Ani is a type of animal."}, {"id": 223, "ent": "buyer", "catw": "material", "stmt": "Buyer is a type of material."}, {"id": 224, "ent": "bankrupt", "catw": "man-made object", "stmt": "Bankrupt is a type of man-made object."}, {"id": 225, "ent": "landlady", "catw": "flower", "stmt": "Landlady is a type of flower."}, {"id": 226, "ent": "winger", "catw": "aircraft", "stmt": "Winger is a type of aircraft."}, {"id": 227, "ent": "meatball", "catw": "food", "stmt": "Meatball is a type of food."}, {"id": 228, "ent": "autofocus", "catw": "man-made object", "stmt": "Autofocus is a type of man-made object."}, {"id": 229, "ent": "narrator", "catw": "clothing", "stmt": "Narrator is a type of clothing."}, {"id": 230, "ent": "winch", "catw": "musical instrument", "stmt": "Winch is a type of musical instrument."}, {"id": 231, "ent": "hoist", "catw": "car", "stmt": "Hoist is a type of car."}, {"id": 232, "ent": "poodle", "catw": "disease", "stmt": "Poodle is a type of disease."}, {"id": 233, "ent": "hamster", "catw": "athlete", "stmt": "Hamster is a type of athlete."}, {"id": 234, "ent": "finch", "catw": "bird", "stmt": "Finch is a type of bird."}, {"id": 235, "ent": "shrine", "catw": "vehicle", "stmt": "Shrine is a type of vehicle."}, {"id": 236, "ent": "ivy", "catw": "car", "stmt": "Ivy is a type of car."}, {"id": 237, "ent": "pensioner", "catw": "person", "stmt": "Pensioner is a type of person."}, {"id": 238, "ent": "flatmate", "catw": "organ", "stmt": "Flatmate is a type of organ."}, {"id": 239, "ent": "communicator", "catw": "living thing", "stmt": "Communicator is a type of living thing."}, {"id": 240, "ent": "menial", "catw": "plant", "stmt": "Menial is a type of plant."}, {"id": 241, "ent": "grandparent", "catw": "food", "stmt": "Grandparent is a type of food."}, {"id": 242, "ent": "chalet", "catw": "building", "stmt": "Chalet is a type of building."}, {"id": 243, "ent": "avenger", "catw": "vegetable", "stmt": "Avenger is a type of vegetable."}, {"id": 244, "ent": "gazebo", "catw": "fish", "stmt": "Gazebo is a type of fish."}, {"id": 245, "ent": "marten", "catw": "living thing", "stmt": "Marten is a type of living thing."}, {"id": 246, "ent": "uterus", "catw": "container", "stmt": "Uterus is a type of container."}, {"id": 247, "ent": "mockingbird", "catw": "person", "stmt": "Mockingbird is a type of person."}, {"id": 248, "ent": "milkman", "catw": "living thing", "stmt": "Milkman is a type of living thing."}, {"id": 249, "ent": "sauna", "catw": "animal", "stmt": "Sauna is a type of animal."}, {"id": 250, "ent": "aviator", "catw": "man-made object", "stmt": "Aviator is a type of man-made object."}, {"id": 251, "ent": "freeman", "catw": "worker", "stmt": "Freeman is a type of worker."}, {"id": 252, "ent": "salesman", "catw": "fish", "stmt": "Salesman is a type of fish."}, {"id": 253, "ent": "synchrotron", "catw": "device", "stmt": "Synchrotron is a type of device."}, {"id": 254, "ent": "traveler", "catw": "reptile", "stmt": "Traveler is a type of reptile."}, {"id": 255, "ent": "skewer", "catw": "game", "stmt": "Skewer is a type of game."}, {"id": 256, "ent": "sleuth", "catw": "metal", "stmt": "Sleuth is a type of metal."}, {"id": 257, "ent": "abolitionist", "catw": "artist", "stmt": "Abolitionist is a type of artist."}, {"id": 258, "ent": "mantel", "catw": "device", "stmt": "Mantel is a type of device."}, {"id": 259, "ent": "consulate", "catw": "building", "stmt": "Consulate is a type of building."}, {"id": 260, "ent": "minster", "catw": "boat", "stmt": "Minster is a type of boat."}, {"id": 261, "ent": "informer", "catw": "fruit", "stmt": "Informer is a type of fruit."}, {"id": 262, "ent": "waitress", "catw": "food", "stmt": "Waitress is a type of food."}, {"id": 263, "ent": "loudspeaker", "catw": "man-made object", "stmt": "Loudspeaker is a type of man-made object."}, {"id": 264, "ent": "orator", "catw": "footwear", "stmt": "Orator is a type of footwear."}, {"id": 265, "ent": "drummer", "catw": "worker", "stmt": "Drummer is a type of worker."}, {"id": 266, "ent": "dingo", "catw": "animal", "stmt": "Dingo is a type of animal."}, {"id": 267, "ent": "sailboat", "catw": "machine", "stmt": "Sailboat is a type of machine."}, {"id": 268, "ent": "idealist", "catw": "disease", "stmt": "Idealist is a type of disease."}, {"id": 269, "ent": "cactus", "catw": "weapon", "stmt": "Cactus is a type of weapon."}, {"id": 270, "ent": "basset", "catw": "animal", "stmt": "Basset is a type of animal."}, {"id": 271, "ent": "handkerchief", "catw": "man-made object", "stmt": "Handkerchief is a type of man-made object."}, {"id": 272, "ent": "plumber", "catw": "tree", "stmt": "Plumber is a type of tree."}, {"id": 273, "ent": "webcam", "catw": "man-made object", "stmt": "Webcam is a type of man-made object."}, {"id": 274, "ent": "dinghy", "catw": "boat", "stmt": "Dinghy is a type of boat."}, {"id": 275, "ent": "mod", "catw": "metal", "stmt": "Mod is a type of metal."}, {"id": 276, "ent": "erotic", "catw": "flower", "stmt": "Erotic is a type of flower."}, {"id": 277, "ent": "songbird", "catw": "room", "stmt": "Songbird is a type of room."}, {"id": 278, "ent": "sauerkraut", "catw": "fruit", "stmt": "Sauerkraut is a type of fruit."}, {"id": 279, "ent": "plenipotentiary", "catw": "equipment", "stmt": "Plenipotentiary is a type of equipment."}, {"id": 280, "ent": "dispensary", "catw": "man-made object", "stmt": "Dispensary is a type of man-made object."}, {"id": 281, "ent": "fetus", "catw": "man-made object", "stmt": "Fetus is a type of man-made object."}, {"id": 282, "ent": "orchid", "catw": "artist", "stmt": "Orchid is a type of artist."}, {"id": 283, "ent": "basset", "catw": "dog", "stmt": "Basset is a type of dog."}, {"id": 284, "ent": "coordinator", "catw": "person", "stmt": "Coordinator is a type of person."}, {"id": 285, "ent": "taster", "catw": "worker", "stmt": "Taster is a type of worker."}, {"id": 286, "ent": "swot", "catw": "sport", "stmt": "Swot is a type of sport."}, {"id": 287, "ent": "exporter", "catw": "material", "stmt": "Exporter is a type of material."}, {"id": 288, "ent": "underwear", "catw": "clothing", "stmt": "Underwear is a type of clothing."}, {"id": 289, "ent": "pendulum", "catw": "device", "stmt": "Pendulum is a type of device."}, {"id": 290, "ent": "stoner", "catw": "bird", "stmt": "Stoner is a type of bird."}, {"id": 291, "ent": "bartender", "catw": "game", "stmt": "Bartender is a type of game."}, {"id": 292, "ent": "juggler", "catw": "living thing", "stmt": "Juggler is a type of living thing."}, {"id": 293, "ent": "sovereign", "catw": "game", "stmt": "Sovereign is a type of game."}, {"id": 294, "ent": "stepson", "catw": "mammal", "stmt": "Stepson is a type of mammal."}, {"id": 295, "ent": "mongoose", "catw": "living thing", "stmt": "Mongoose is a type of living thing."}, {"id": 296, "ent": "hater", "catw": "dog", "stmt": "Hater is a type of dog."}, {"id": 297, "ent": "cutthroat", "catw": "game", "stmt": "Cutthroat is a type of game."}, {"id": 298, "ent": "pedophile", "catw": "food", "stmt": "Pedophile is a type of food."}, {"id": 299, "ent": "ripper", "catw": "plant", "stmt": "Ripper is a type of plant."}, {"id": 300, "ent": "sultan", "catw": "organ", "stmt": "Sultan is a type of organ."}, {"id": 301, "ent": "assistant", "catw": "dish", "stmt": "Assistant is a type of dish."}, {"id": 302, "ent": "coauthor", "catw": "vegetable", "stmt": "Coauthor is a type of vegetable."}, {"id": 303, "ent": "ladybug", "catw": "equipment", "stmt": "Ladybug is a type of equipment."}, {"id": 304, "ent": "dredge", "catw": "man-made object", "stmt": "Dredge is a type of man-made object."}, {"id": 305, "ent": "clarence", "catw": "vehicle", "stmt": "Clarence is a type of vehicle."}, {"id": 306, "ent": "eft", "catw": "aircraft", "stmt": "Eft is a type of aircraft."}, {"id": 307, "ent": "modernist", "catw": "building", "stmt": "Modernist is a type of building."}, {"id": 308, "ent": "satchel", "catw": "man-made object", "stmt": "Satchel is a type of man-made object."}, {"id": 309, "ent": "invalid", "catw": "game", "stmt": "Invalid is a type of game."}, {"id": 310, "ent": "jogger", "catw": "food", "stmt": "Jogger is a type of food."}, {"id": 311, "ent": "rancher", "catw": "body part", "stmt": "Rancher is a type of body part."}, {"id": 312, "ent": "clary", "catw": "body part", "stmt": "Clary is a type of body part."}, {"id": 313, "ent": "recliner", "catw": "body part", "stmt": "Recliner is a type of body part."}, {"id": 314, "ent": "mercenary", "catw": "worker", "stmt": "Mercenary is a type of worker."}, {"id": 315, "ent": "storeroom", "catw": "vegetable", "stmt": "Storeroom is a type of vegetable."}, {"id": 316, "ent": "protege", "catw": "vegetable", "stmt": "Protege is a type of vegetable."}, {"id": 317, "ent": "handyman", "catw": "weapon", "stmt": "Handyman is a type of weapon."}, {"id": 318, "ent": "wanton", "catw": "living thing", "stmt": "Wanton is a type of living thing."}, {"id": 319, "ent": "bloodhound", "catw": "vegetable", "stmt": "Bloodhound is a type of vegetable."}, {"id": 320, "ent": "crustacean", "catw": "body part", "stmt": "Crustacean is a type of body part."}, {"id": 321, "ent": "electric", "catw": "car", "stmt": "Electric is a type of car."}, {"id": 322, "ent": "itinerant", "catw": "artist", "stmt": "Itinerant is a type of artist."}, {"id": 323, "ent": "outhouse", "catw": "container", "stmt": "Outhouse is a type of container."}, {"id": 324, "ent": "searchlight", "catw": "metal", "stmt": "Searchlight is a type of metal."}, {"id": 325, "ent": "stickler", "catw": "person", "stmt": "Stickler is a type of person."}, {"id": 326, "ent": "snob", "catw": "person", "stmt": "Snob is a type of person."}, {"id": 327, "ent": "erotic", "catw": "living thing", "stmt": "Erotic is a type of living thing."}, {"id": 328, "ent": "violin", "catw": "musical instrument", "stmt": "Violin is a type of musical instrument."}, {"id": 329, "ent": "plover", "catw": "bird", "stmt": "Plover is a type of bird."}, {"id": 330, "ent": "lobbyist", "catw": "boat", "stmt": "Lobbyist is a type of boat."}, {"id": 331, "ent": "motorcyclist", "catw": "footwear", "stmt": "Motorcyclist is a type of footwear."}, {"id": 332, "ent": "harlequin", "catw": "reptile", "stmt": "Harlequin is a type of reptile."}, {"id": 333, "ent": "cheeseburger", "catw": "man-made object", "stmt": "Cheeseburger is a type of man-made object."}, {"id": 334, "ent": "wearer", "catw": "mammal", "stmt": "Wearer is a type of mammal."}, {"id": 335, "ent": "geisha", "catw": "plant", "stmt": "Geisha is a type of plant."}, {"id": 336, "ent": "poppy", "catw": "sport", "stmt": "Poppy is a type of sport."}, {"id": 337, "ent": "bobbin", "catw": "fruit", "stmt": "Bobbin is a type of fruit."}, {"id": 338, "ent": "preacher", "catw": "person", "stmt": "Preacher is a type of person."}, {"id": 339, "ent": "ladybug", "catw": "insect", "stmt": "Ladybug is a type of insect."}, {"id": 340, "ent": "cubby", "catw": "living thing", "stmt": "Cubby is a type of living thing."}, {"id": 341, "ent": "neurotic", "catw": "metal", "stmt": "Neurotic is a type of metal."}, {"id": 342, "ent": "applicator", "catw": "device", "stmt": "Applicator is a type of device."}, {"id": 343, "ent": "hoarder", "catw": "mammal", "stmt": "Hoarder is a type of mammal."}, {"id": 344, "ent": "escapist", "catw": "game", "stmt": "Escapist is a type of game."}, {"id": 345, "ent": "bereaved", "catw": "living thing", "stmt": "Bereaved is a type of living thing."}, {"id": 346, "ent": "termite", "catw": "mammal", "stmt": "Termite is a type of mammal."}, {"id": 347, "ent": "dragonfly", "catw": "fish", "stmt": "Dragonfly is a type of fish."}, {"id": 348, "ent": "razor", "catw": "weapon", "stmt": "Razor is a type of weapon."}, {"id": 349, "ent": "mensch", "catw": "person", "stmt": "Mensch is a type of person."}, {"id": 350, "ent": "kingfisher", "catw": "insect", "stmt": "Kingfisher is a type of insect."}, {"id": 351, "ent": "snob", "catw": "vegetable", "stmt": "Snob is a type of vegetable."}, {"id": 352, "ent": "bock", "catw": "vegetable", "stmt": "Bock is a type of vegetable."}, {"id": 353, "ent": "accelerometer", "catw": "sport", "stmt": "Accelerometer is a type of sport."}, {"id": 354, "ent": "electrode", "catw": "equipment", "stmt": "Electrode is a type of equipment."}, {"id": 355, "ent": "subcontractor", "catw": "food", "stmt": "Subcontractor is a type of food."}, {"id": 356, "ent": "vase", "catw": "container", "stmt": "Vase is a type of container."}, {"id": 357, "ent": "parachute", "catw": "equipment", "stmt": "Parachute is a type of equipment."}, {"id": 358, "ent": "soloist", "catw": "metal", "stmt": "Soloist is a type of metal."}, {"id": 359, "ent": "prankster", "catw": "person", "stmt": "Prankster is a type of person."}, {"id": 360, "ent": "missus", "catw": "person", "stmt": "Missus is a type of person."}, {"id": 361, "ent": "gynecologist", "catw": "living thing", "stmt": "Gynecologist is a type of living thing."}, {"id": 362, "ent": "landowner", "catw": "person", "stmt": "Landowner is a type of person."}, {"id": 363, "ent": "mosquito", "catw": "mammal", "stmt": "Mosquito is a type of mammal."}, {"id": 364, "ent": "decadent", "catw": "dish", "stmt": "Decadent is a type of dish."}, {"id": 365, "ent": "toolbox", "catw": "tool", "stmt": "Toolbox is a type of tool."}, {"id": 366, "ent": "brandy", "catw": "food", "stmt": "Brandy is a type of food."}, {"id": 367, "ent": "housekeeper", "catw": "flower", "stmt": "Housekeeper is a type of flower."}, {"id": 368, "ent": "convener", "catw": "material", "stmt": "Convener is a type of material."}, {"id": 369, "ent": "moa", "catw": "animal", "stmt": "Moa is a type of animal."}, {"id": 370, "ent": "advertiser", "catw": "metal", "stmt": "Advertiser is a type of metal."}, {"id": 371, "ent": "spaniel", "catw": "dog", "stmt": "Spaniel is a type of dog."}, {"id": 372, "ent": "denture", "catw": "man-made object", "stmt": "Denture is a type of man-made object."}, {"id": 373, "ent": "pigeon", "catw": "person", "stmt": "Pigeon is a type of person."}, {"id": 374, "ent": "chrysalis", "catw": "body part", "stmt": "Chrysalis is a type of body part."}, {"id": 375, "ent": "vizier", "catw": "athlete", "stmt": "Vizier is a type of athlete."}, {"id": 376, "ent": "bandit", "catw": "device", "stmt": "Bandit is a type of device."}, {"id": 377, "ent": "dominatrix", "catw": "living thing", "stmt": "Dominatrix is a type of living thing."}, {"id": 378, "ent": "protectionist", "catw": "plant", "stmt": "Protectionist is a type of plant."}, {"id": 379, "ent": "cyborg", "catw": "body part", "stmt": "Cyborg is a type of body part."}, {"id": 380, "ent": "manger", "catw": "worker", "stmt": "Manger is a type of worker."}, {"id": 381, "ent": "stowaway", "catw": "vegetable", "stmt": "Stowaway is a type of vegetable."}, {"id": 382, "ent": "stingray", "catw": "living thing", "stmt": "Stingray is a type of living thing."}, {"id": 383, "ent": "immigrant", "catw": "game", "stmt": "Immigrant is a type of game."}, {"id": 384, "ent": "peripheral", "catw": "animal", "stmt": "Peripheral is a type of animal."}, {"id": 385, "ent": "corgi", "catw": "clothing", "stmt": "Corgi is a type of clothing."}, {"id": 386, "ent": "trusty", "catw": "game", "stmt": "Trusty is a type of game."}, {"id": 387, "ent": "inventor", "catw": "drink", "stmt": "Inventor is a type of drink."}, {"id": 388, "ent": "narc", "catw": "person", "stmt": "Narc is a type of person."}, {"id": 389, "ent": "brazier", "catw": "drink", "stmt": "Brazier is a type of drink."}, {"id": 390, "ent": "curate", "catw": "living thing", "stmt": "Curate is a type of living thing."}, {"id": 391, "ent": "stagecoach", "catw": "vehicle", "stmt": "Stagecoach is a type of vehicle."}, {"id": 392, "ent": "molester", "catw": "mammal", "stmt": "Molester is a type of mammal."}, {"id": 393, "ent": "paramedic", "catw": "worker", "stmt": "Paramedic is a type of worker."}, {"id": 394, "ent": "inventor", "catw": "body part", "stmt": "Inventor is a type of body part."}, {"id": 395, "ent": "synagogue", "catw": "vegetable", "stmt": "Synagogue is a type of vegetable."}, {"id": 396, "ent": "adolescent", "catw": "mammal", "stmt": "Adolescent is a type of mammal."}, {"id": 397, "ent": "contrarian", "catw": "metal", "stmt": "Contrarian is a type of metal."}, {"id": 398, "ent": "swimsuit", "catw": "equipment", "stmt": "Swimsuit is a type of equipment."}, {"id": 399, "ent": "shrine", "catw": "container", "stmt": "Shrine is a type of container."}, {"id": 400, "ent": "manatee", "catw": "vehicle", "stmt": "Manatee is a type of vehicle."}, {"id": 401, "ent": "toddler", "catw": "tool", "stmt": "Toddler is a type of tool."}, {"id": 402, "ent": "introvert", "catw": "worker", "stmt": "Introvert is a type of worker."}, {"id": 403, "ent": "copywriter", "catw": "clothing", "stmt": "Copywriter is a type of clothing."}, {"id": 404, "ent": "vigilante", "catw": "person", "stmt": "Vigilante is a type of person."}, {"id": 405, "ent": "cuttlefish", "catw": "animal", "stmt": "Cuttlefish is a type of animal."}, {"id": 406, "ent": "alderman", "catw": "athlete", "stmt": "Alderman is a type of athlete."}, {"id": 407, "ent": "hater", "catw": "boat", "stmt": "Hater is a type of boat."}, {"id": 408, "ent": "airship", "catw": "dog", "stmt": "Airship is a type of dog."}, {"id": 409, "ent": "benefactor", "catw": "person", "stmt": "Benefactor is a type of person."}, {"id": 410, "ent": "financier", "catw": "worker", "stmt": "Financier is a type of worker."}, {"id": 411, "ent": "trader", "catw": "worker", "stmt": "Trader is a type of worker."}, {"id": 412, "ent": "anti", "catw": "living thing", "stmt": "Anti is a type of living thing."}, {"id": 413, "ent": "sarcophagus", "catw": "tool", "stmt": "Sarcophagus is a type of tool."}, {"id": 414, "ent": "paleontologist", "catw": "worker", "stmt": "Paleontologist is a type of worker."}, {"id": 415, "ent": "accordion", "catw": "fish", "stmt": "Accordion is a type of fish."}, {"id": 416, "ent": "sphincter", "catw": "disease", "stmt": "Sphincter is a type of disease."}, {"id": 417, "ent": "detainee", "catw": "living thing", "stmt": "Detainee is a type of living thing."}, {"id": 418, "ent": "pleaser", "catw": "game", "stmt": "Pleaser is a type of game."}, {"id": 419, "ent": "princess", "catw": "person", "stmt": "Princess is a type of person."}, {"id": 420, "ent": "paramedic", "catw": "dish", "stmt": "Paramedic is a type of dish."}, {"id": 421, "ent": "rink", "catw": "sport", "stmt": "Rink is a type of sport."}, {"id": 422, "ent": "propagandist", "catw": "person", "stmt": "Propagandist is a type of person."}, {"id": 423, "ent": "confidante", "catw": "person", "stmt": "Confidante is a type of person."}, {"id": 424, "ent": "stockbroker", "catw": "footwear", "stmt": "Stockbroker is a type of footwear."}, {"id": 425, "ent": "drosophila", "catw": "food", "stmt": "Drosophila is a type of food."}, {"id": 426, "ent": "pacifist", "catw": "insect", "stmt": "Pacifist is a type of insect."}, {"id": 427, "ent": "suitor", "catw": "game", "stmt": "Suitor is a type of game."}, {"id": 428, "ent": "apologist", "catw": "worker", "stmt": "Apologist is a type of worker."}, {"id": 429, "ent": "clubhouse", "catw": "room", "stmt": "Clubhouse is a type of room."}, {"id": 430, "ent": "strategist", "catw": "person", "stmt": "Strategist is a type of person."}, {"id": 431, "ent": "jeep", "catw": "container", "stmt": "Jeep is a type of container."}, {"id": 432, "ent": "survivalist", "catw": "fish", "stmt": "Survivalist is a type of fish."}, {"id": 433, "ent": "vermouth", "catw": "plant", "stmt": "Vermouth is a type of plant."}, {"id": 434, "ent": "contrarian", "catw": "animal", "stmt": "Contrarian is a type of animal."}, {"id": 435, "ent": "cattle", "catw": "animal", "stmt": "Cattle is a type of animal."}, {"id": 436, "ent": "pensioner", "catw": "sport", "stmt": "Pensioner is a type of sport."}, {"id": 437, "ent": "marlin", "catw": "living thing", "stmt": "Marlin is a type of living thing."}, {"id": 438, "ent": "mockingbird", "catw": "bird", "stmt": "Mockingbird is a type of bird."}, {"id": 439, "ent": "dandelion", "catw": "mammal", "stmt": "Dandelion is a type of mammal."}, {"id": 440, "ent": "meatball", "catw": "game", "stmt": "Meatball is a type of game."}, {"id": 441, "ent": "chime", "catw": "organ", "stmt": "Chime is a type of organ."}, {"id": 442, "ent": "hornet", "catw": "worker", "stmt": "Hornet is a type of worker."}, {"id": 443, "ent": "wildflower", "catw": "living thing", "stmt": "Wildflower is a type of living thing."}, {"id": 444, "ent": "gyroscope", "catw": "machine", "stmt": "Gyroscope is a type of machine."}, {"id": 445, "ent": "lender", "catw": "bird", "stmt": "Lender is a type of bird."}, {"id": 446, "ent": "kangaroo", "catw": "athlete", "stmt": "Kangaroo is a type of athlete."}, {"id": 447, "ent": "calico", "catw": "fabric", "stmt": "Calico is a type of fabric."}, {"id": 448, "ent": "tongs", "catw": "tool", "stmt": "Tongs is a type of tool."}, {"id": 449, "ent": "cadet", "catw": "dish", "stmt": "Cadet is a type of dish."}, {"id": 450, "ent": "repairman", "catw": "fruit", "stmt": "Repairman is a type of fruit."}, {"id": 451, "ent": "clubhouse", "catw": "machine", "stmt": "Clubhouse is a type of machine."}, {"id": 452, "ent": "hummingbird", "catw": "animal", "stmt": "Hummingbird is a type of animal."}, {"id": 453, "ent": "bison", "catw": "mammal", "stmt": "Bison is a type of mammal."}, {"id": 454, "ent": "piglet", "catw": "mammal", "stmt": "Piglet is a type of mammal."}, {"id": 455, "ent": "hyena", "catw": "plant", "stmt": "Hyena is a type of plant."}, {"id": 456, "ent": "wife", "catw": "mammal", "stmt": "Wife is a type of mammal."}, {"id": 457, "ent": "berk", "catw": "living thing", "stmt": "Berk is a type of living thing."}, {"id": 458, "ent": "transistor", "catw": "fish", "stmt": "Transistor is a type of fish."}, {"id": 459, "ent": "firstborn", "catw": "sport", "stmt": "Firstborn is a type of sport."}, {"id": 460, "ent": "interviewee", "catw": "worker", "stmt": "Interviewee is a type of worker."}, {"id": 461, "ent": "surrealist", "catw": "fish", "stmt": "Surrealist is a type of fish."}, {"id": 462, "ent": "corrective", "catw": "man-made object", "stmt": "Corrective is a type of man-made object."}, {"id": 463, "ent": "searchlight", "catw": "equipment", "stmt": "Searchlight is a type of equipment."}, {"id": 464, "ent": "bookkeeper", "catw": "reptile", "stmt": "Bookkeeper is a type of reptile."}, {"id": 465, "ent": "takin", "catw": "fish", "stmt": "Takin is a type of fish."}, {"id": 466, "ent": "therapist", "catw": "living thing", "stmt": "Therapist is a type of living thing."}, {"id": 467, "ent": "communicator", "catw": "person", "stmt": "Communicator is a type of person."}, {"id": 468, "ent": "bereaved", "catw": "game", "stmt": "Bereaved is a type of game."}, {"id": 469, "ent": "apparel", "catw": "weapon", "stmt": "Apparel is a type of weapon."}, {"id": 470, "ent": "boardroom", "catw": "room", "stmt": "Boardroom is a type of room."}, {"id": 471, "ent": "goldilocks", "catw": "flower", "stmt": "Goldilocks is a type of flower."}, {"id": 472, "ent": "manatee", "catw": "food", "stmt": "Manatee is a type of food."}, {"id": 473, "ent": "aloe", "catw": "room", "stmt": "Aloe is a type of room."}, {"id": 474, "ent": "congressman", "catw": "flower", "stmt": "Congressman is a type of flower."}, {"id": 475, "ent": "ecologist", "catw": "worker", "stmt": "Ecologist is a type of worker."}, {"id": 476, "ent": "dahlia", "catw": "plant", "stmt": "Dahlia is a type of plant."}, {"id": 477, "ent": "lodger", "catw": "plant", "stmt": "Lodger is a type of plant."}, {"id": 478, "ent": "flea", "catw": "animal", "stmt": "Flea is a type of animal."}, {"id": 479, "ent": "hillbilly", "catw": "living thing", "stmt": "Hillbilly is a type of living thing."}, {"id": 480, "ent": "stewardess", "catw": "equipment", "stmt": "Stewardess is a type of equipment."}, {"id": 481, "ent": "helmsman", "catw": "athlete", "stmt": "Helmsman is a type of athlete."}, {"id": 482, "ent": "jasmine", "catw": "vegetable", "stmt": "Jasmine is a type of vegetable."}, {"id": 483, "ent": "palmetto", "catw": "musical instrument", "stmt": "Palmetto is a type of musical instrument."}, {"id": 484, "ent": "rapporteur", "catw": "organ", "stmt": "Rapporteur is a type of organ."}, {"id": 485, "ent": "bobcat", "catw": "mammal", "stmt": "Bobcat is a type of mammal."}, {"id": 486, "ent": "cutler", "catw": "living thing", "stmt": "Cutler is a type of living thing."}, {"id": 487, "ent": "hearse", "catw": "man-made object", "stmt": "Hearse is a type of man-made object."}, {"id": 488, "ent": "spastic", "catw": "reptile", "stmt": "Spastic is a type of reptile."}, {"id": 489, "ent": "alcoholic", "catw": "drink", "stmt": "Alcoholic is a type of drink."}, {"id": 490, "ent": "longhorn", "catw": "tree", "stmt": "Longhorn is a type of tree."}, {"id": 491, "ent": "beagle", "catw": "sport", "stmt": "Beagle is a type of sport."}, {"id": 492, "ent": "impressionist", "catw": "fabric", "stmt": "Impressionist is a type of fabric."}, {"id": 493, "ent": "inpatient", "catw": "fruit", "stmt": "Inpatient is a type of fruit."}, {"id": 494, "ent": "emir", "catw": "boat", "stmt": "Emir is a type of boat."}, {"id": 495, "ent": "snapdragon", "catw": "living thing", "stmt": "Snapdragon is a type of living thing."}, {"id": 496, "ent": "stockholder", "catw": "furniture", "stmt": "Stockholder is a type of furniture."}, {"id": 497, "ent": "schizophrenic", "catw": "sport", "stmt": "Schizophrenic is a type of sport."}, {"id": 498, "ent": "rapporteur", "catw": "person", "stmt": "Rapporteur is a type of person."}, {"id": 499, "ent": "ombudsman", "catw": "living thing", "stmt": "Ombudsman is a type of living thing."}, {"id": 500, "ent": "cleric", "catw": "plant", "stmt": "Cleric is a type of plant."}, {"id": 501, "ent": "patriot", "catw": "aircraft", "stmt": "Patriot is a type of aircraft."}, {"id": 502, "ent": "homophobe", "catw": "sport", "stmt": "Homophobe is a type of sport."}, {"id": 503, "ent": "rifle", "catw": "animal", "stmt": "Rifle is a type of animal."}, {"id": 504, "ent": "phytoplankton", "catw": "plant", "stmt": "Phytoplankton is a type of plant."}, {"id": 505, "ent": "caveman", "catw": "person", "stmt": "Caveman is a type of person."}, {"id": 506, "ent": "exporter", "catw": "drink", "stmt": "Exporter is a type of drink."}, {"id": 507, "ent": "sibling", "catw": "mammal", "stmt": "Sibling is a type of mammal."}, {"id": 508, "ent": "limousine", "catw": "clothing", "stmt": "Limousine is a type of clothing."}, {"id": 509, "ent": "lyre", "catw": "man-made object", "stmt": "Lyre is a type of man-made object."}, {"id": 510, "ent": "audiovisual", "catw": "material", "stmt": "Audiovisual is a type of material."}, {"id": 511, "ent": "communicator", "catw": "sport", "stmt": "Communicator is a type of sport."}, {"id": 512, "ent": "bathrobe", "catw": "metal", "stmt": "Bathrobe is a type of metal."}, {"id": 513, "ent": "thwart", "catw": "device", "stmt": "Thwart is a type of device."}, {"id": 514, "ent": "stork", "catw": "aircraft", "stmt": "Stork is a type of aircraft."}, {"id": 515, "ent": "marksman", "catw": "game", "stmt": "Marksman is a type of game."}, {"id": 516, "ent": "confessional", "catw": "toy", "stmt": "Confessional is a type of toy."}, {"id": 517, "ent": "monastery", "catw": "container", "stmt": "Monastery is a type of container."}, {"id": 518, "ent": "headdress", "catw": "flower", "stmt": "Headdress is a type of flower."}, {"id": 519, "ent": "statesman", "catw": "living thing", "stmt": "Statesman is a type of living thing."}, {"id": 520, "ent": "milkman", "catw": "vegetable", "stmt": "Milkman is a type of vegetable."}, {"id": 521, "ent": "thermometer", "catw": "tool", "stmt": "Thermometer is a type of tool."}, {"id": 522, "ent": "expressionist", "catw": "game", "stmt": "Expressionist is a type of game."}, {"id": 523, "ent": "perpetrator", "catw": "insect", "stmt": "Perpetrator is a type of insect."}, {"id": 524, "ent": "coupe", "catw": "container", "stmt": "Coupe is a type of container."}, {"id": 525, "ent": "councilman", "catw": "worker", "stmt": "Councilman is a type of worker."}, {"id": 526, "ent": "authoritarian", "catw": "sport", "stmt": "Authoritarian is a type of sport."}, {"id": 527, "ent": "satirist", "catw": "artist", "stmt": "Satirist is a type of artist."}, {"id": 528, "ent": "antelope", "catw": "living thing", "stmt": "Antelope is a type of living thing."}, {"id": 529, "ent": "scraper", "catw": "device", "stmt": "Scraper is a type of device."}, {"id": 530, "ent": "ballroom", "catw": "sport", "stmt": "Ballroom is a type of sport."}, {"id": 531, "ent": "dunce", "catw": "flower", "stmt": "Dunce is a type of flower."}, {"id": 532, "ent": "hiker", "catw": "athlete", "stmt": "Hiker is a type of athlete."}, {"id": 533, "ent": "playmate", "catw": "mammal", "stmt": "Playmate is a type of mammal."}, {"id": 534, "ent": "agitator", "catw": "vegetable", "stmt": "Agitator is a type of vegetable."}, {"id": 535, "ent": "stiletto", "catw": "footwear", "stmt": "Stiletto is a type of footwear."}, {"id": 536, "ent": "clock", "catw": "machine", "stmt": "Clock is a type of machine."}, {"id": 537, "ent": "bankrupt", "catw": "person", "stmt": "Bankrupt is a type of person."}, {"id": 538, "ent": "sophomore", "catw": "boat", "stmt": "Sophomore is a type of boat."}, {"id": 539, "ent": "hillbilly", "catw": "material", "stmt": "Hillbilly is a type of material."}, {"id": 540, "ent": "yarrow", "catw": "flower", "stmt": "Yarrow is a type of flower."}, {"id": 541, "ent": "caveman", "catw": "living thing", "stmt": "Caveman is a type of living thing."}, {"id": 542, "ent": "battleship", "catw": "person", "stmt": "Battleship is a type of person."}, {"id": 543, "ent": "ratchet", "catw": "tool", "stmt": "Ratchet is a type of tool."}, {"id": 544, "ent": "rival", "catw": "weapon", "stmt": "Rival is a type of weapon."}, {"id": 545, "ent": "candlestick", "catw": "device", "stmt": "Candlestick is a type of device."}, {"id": 546, "ent": "helmsman", "catw": "boat", "stmt": "Helmsman is a type of boat."}, {"id": 547, "ent": "suitor", "catw": "person", "stmt": "Suitor is a type of person."}, {"id": 548, "ent": "greenhouse", "catw": "device", "stmt": "Greenhouse is a type of device."}, {"id": 549, "ent": "socialist", "catw": "person", "stmt": "Socialist is a type of person."}, {"id": 550, "ent": "prince", "catw": "person", "stmt": "Prince is a type of person."}, {"id": 551, "ent": "pulley", "catw": "footwear", "stmt": "Pulley is a type of footwear."}, {"id": 552, "ent": "parachute", "catw": "device", "stmt": "Parachute is a type of device."}, {"id": 553, "ent": "crocodile", "catw": "body part", "stmt": "Crocodile is a type of body part."}, {"id": 554, "ent": "innocent", "catw": "man-made object", "stmt": "Innocent is a type of man-made object."}, {"id": 555, "ent": "conch", "catw": "living thing", "stmt": "Conch is a type of living thing."}, {"id": 556, "ent": "electrode", "catw": "car", "stmt": "Electrode is a type of car."}, {"id": 557, "ent": "raglan", "catw": "furniture", "stmt": "Raglan is a type of furniture."}, {"id": 558, "ent": "nacho", "catw": "material", "stmt": "Nacho is a type of material."}, {"id": 559, "ent": "surgeon", "catw": "worker", "stmt": "Surgeon is a type of worker."}, {"id": 560, "ent": "artist", "catw": "fish", "stmt": "Artist is a type of fish."}, {"id": 561, "ent": "claimant", "catw": "disease", "stmt": "Claimant is a type of disease."}, {"id": 562, "ent": "ibis", "catw": "footwear", "stmt": "Ibis is a type of footwear."}, {"id": 563, "ent": "notary", "catw": "person", "stmt": "Notary is a type of person."}, {"id": 564, "ent": "motorboat", "catw": "machine", "stmt": "Motorboat is a type of machine."}, {"id": 565, "ent": "loom", "catw": "man-made object", "stmt": "Loom is a type of man-made object."}, {"id": 566, "ent": "firearm", "catw": "weapon", "stmt": "Firearm is a type of weapon."}, {"id": 567, "ent": "haggis", "catw": "organ", "stmt": "Haggis is a type of organ."}, {"id": 568, "ent": "brogan", "catw": "food", "stmt": "Brogan is a type of food."}, {"id": 569, "ent": "tam", "catw": "dish", "stmt": "Tam is a type of dish."}, {"id": 570, "ent": "motorcyclist", "catw": "sport", "stmt": "Motorcyclist is a type of sport."}, {"id": 571, "ent": "alumnus", "catw": "plant", "stmt": "Alumnus is a type of plant."}, {"id": 572, "ent": "dildo", "catw": "equipment", "stmt": "Dildo is a type of equipment."}, {"id": 573, "ent": "tights", "catw": "vegetable", "stmt": "Tights is a type of vegetable."}, {"id": 574, "ent": "doorbell", "catw": "boat", "stmt": "Doorbell is a type of boat."}, {"id": 575, "ent": "letterman", "catw": "reptile", "stmt": "Letterman is a type of reptile."}, {"id": 576, "ent": "athlete", "catw": "worker", "stmt": "Athlete is a type of worker."}, {"id": 577, "ent": "hostage", "catw": "game", "stmt": "Hostage is a type of game."}, {"id": 578, "ent": "empty", "catw": "container", "stmt": "Empty is a type of container."}, {"id": 579, "ent": "spacecraft", "catw": "machine", "stmt": "Spacecraft is a type of machine."}, {"id": 580, "ent": "kilt", "catw": "boat", "stmt": "Kilt is a type of boat."}, {"id": 581, "ent": "shook", "catw": "body part", "stmt": "Shook is a type of body part."}, {"id": 582, "ent": "headscarf", "catw": "clothing", "stmt": "Headscarf is a type of clothing."}, {"id": 583, "ent": "geranium", "catw": "animal", "stmt": "Geranium is a type of animal."}, {"id": 584, "ent": "mangrove", "catw": "weapon", "stmt": "Mangrove is a type of weapon."}, {"id": 585, "ent": "librarian", "catw": "food", "stmt": "Librarian is a type of food."}, {"id": 586, "ent": "restaurateur", "catw": "athlete", "stmt": "Restaurateur is a type of athlete."}, {"id": 587, "ent": "perfectionist", "catw": "living thing", "stmt": "Perfectionist is a type of living thing."}, {"id": 588, "ent": "flashlight", "catw": "tool", "stmt": "Flashlight is a type of tool."}, {"id": 589, "ent": "groundhog", "catw": "man-made object", "stmt": "Groundhog is a type of man-made object."}, {"id": 590, "ent": "contemplative", "catw": "footwear", "stmt": "Contemplative is a type of footwear."}, {"id": 591, "ent": "robber", "catw": "living thing", "stmt": "Robber is a type of living thing."}, {"id": 592, "ent": "tequila", "catw": "plant", "stmt": "Tequila is a type of plant."}, {"id": 593, "ent": "expert", "catw": "person", "stmt": "Expert is a type of person."}, {"id": 594, "ent": "wheelchair", "catw": "game", "stmt": "Wheelchair is a type of game."}, {"id": 595, "ent": "cuttlefish", "catw": "artist", "stmt": "Cuttlefish is a type of artist."}, {"id": 596, "ent": "traveler", "catw": "game", "stmt": "Traveler is a type of game."}, {"id": 597, "ent": "thermos", "catw": "plant", "stmt": "Thermos is a type of plant."}, {"id": 598, "ent": "customer", "catw": "weapon", "stmt": "Customer is a type of weapon."}, {"id": 599, "ent": "loner", "catw": "living thing", "stmt": "Loner is a type of living thing."}, {"id": 600, "ent": "slipknot", "catw": "material", "stmt": "Slipknot is a type of material."}, {"id": 601, "ent": "landlady", "catw": "game", "stmt": "Landlady is a type of game."}, {"id": 602, "ent": "oppressor", "catw": "device", "stmt": "Oppressor is a type of device."}, {"id": 603, "ent": "alderman", "catw": "boat", "stmt": "Alderman is a type of boat."}, {"id": 604, "ent": "fife", "catw": "fabric", "stmt": "Fife is a type of fabric."}, {"id": 605, "ent": "loudspeaker", "catw": "disease", "stmt": "Loudspeaker is a type of disease."}, {"id": 606, "ent": "revisionist", "catw": "artist", "stmt": "Revisionist is a type of artist."}, {"id": 607, "ent": "plenipotentiary", "catw": "mammal", "stmt": "Plenipotentiary is a type of mammal."}, {"id": 608, "ent": "houseboat", "catw": "vehicle", "stmt": "Houseboat is a type of vehicle."}, {"id": 609, "ent": "optimist", "catw": "body part", "stmt": "Optimist is a type of body part."}, {"id": 610, "ent": "primrose", "catw": "flower", "stmt": "Primrose is a type of flower."}, {"id": 611, "ent": "moray", "catw": "insect", "stmt": "Moray is a type of insect."}, {"id": 612, "ent": "roebuck", "catw": "animal", "stmt": "Roebuck is a type of animal."}, {"id": 613, "ent": "cafe", "catw": "man-made object", "stmt": "Cafe is a type of man-made object."}, {"id": 614, "ent": "proletarian", "catw": "worker", "stmt": "Proletarian is a type of worker."}, {"id": 615, "ent": "limiter", "catw": "man-made object", "stmt": "Limiter is a type of man-made object."}, {"id": 616, "ent": "brogan", "catw": "man-made object", "stmt": "Brogan is a type of man-made object."}, {"id": 617, "ent": "wingman", "catw": "game", "stmt": "Wingman is a type of game."}, {"id": 618, "ent": "robber", "catw": "game", "stmt": "Robber is a type of game."}, {"id": 619, "ent": "aquarium", "catw": "boat", "stmt": "Aquarium is a type of boat."}, {"id": 620, "ent": "handkerchief", "catw": "device", "stmt": "Handkerchief is a type of device."}, {"id": 621, "ent": "legislator", "catw": "worker", "stmt": "Legislator is a type of worker."}, {"id": 622, "ent": "preteen", "catw": "person", "stmt": "Preteen is a type of person."}, {"id": 623, "ent": "individualist", "catw": "worker", "stmt": "Individualist is a type of worker."}, {"id": 624, "ent": "detonator", "catw": "furniture", "stmt": "Detonator is a type of furniture."}, {"id": 625, "ent": "banjo", "catw": "fish", "stmt": "Banjo is a type of fish."}, {"id": 626, "ent": "neurosurgeon", "catw": "footwear", "stmt": "Neurosurgeon is a type of footwear."}, {"id": 627, "ent": "sunflower", "catw": "living thing", "stmt": "Sunflower is a type of living thing."}, {"id": 628, "ent": "kylie", "catw": "aircraft", "stmt": "Kylie is a type of aircraft."}, {"id": 629, "ent": "nawab", "catw": "clothing", "stmt": "Nawab is a type of clothing."}, {"id": 630, "ent": "assistant", "catw": "person", "stmt": "Assistant is a type of person."}, {"id": 631, "ent": "cantilever", "catw": "body part", "stmt": "Cantilever is a type of body part."}, {"id": 632, "ent": "libertine", "catw": "fish", "stmt": "Libertine is a type of fish."}, {"id": 633, "ent": "chronicler", "catw": "artist", "stmt": "Chronicler is a type of artist."}, {"id": 634, "ent": "earl", "catw": "boat", "stmt": "Earl is a type of boat."}, {"id": 635, "ent": "abbess", "catw": "building", "stmt": "Abbess is a type of building."}, {"id": 636, "ent": "murine", "catw": "food", "stmt": "Murine is a type of food."}, {"id": 637, "ent": "stole", "catw": "weapon", "stmt": "Stole is a type of weapon."}, {"id": 638, "ent": "stapler", "catw": "boat", "stmt": "Stapler is a type of boat."}, {"id": 639, "ent": "ski", "catw": "device", "stmt": "Ski is a type of device."}, {"id": 640, "ent": "materiel", "catw": "equipment", "stmt": "Materiel is a type of equipment."}, {"id": 641, "ent": "rheumatic", "catw": "living thing", "stmt": "Rheumatic is a type of living thing."}, {"id": 642, "ent": "margarita", "catw": "dish", "stmt": "Margarita is a type of dish."}, {"id": 643, "ent": "dredge", "catw": "animal", "stmt": "Dredge is a type of animal."}, {"id": 644, "ent": "private", "catw": "fruit", "stmt": "Private is a type of fruit."}, {"id": 645, "ent": "chalet", "catw": "car", "stmt": "Chalet is a type of car."}, {"id": 646, "ent": "locket", "catw": "device", "stmt": "Locket is a type of device."}, {"id": 647, "ent": "earl", "catw": "animal", "stmt": "Earl is a type of animal."}, {"id": 648, "ent": "blazer", "catw": "weapon", "stmt": "Blazer is a type of weapon."}, {"id": 649, "ent": "jammer", "catw": "body part", "stmt": "Jammer is a type of body part."}, {"id": 650, "ent": "crustacean", "catw": "food", "stmt": "Crustacean is a type of food."}, {"id": 651, "ent": "android", "catw": "worker", "stmt": "Android is a type of worker."}, {"id": 652, "ent": "cowgirl", "catw": "animal", "stmt": "Cowgirl is a type of animal."}, {"id": 653, "ent": "entrepreneur", "catw": "food", "stmt": "Entrepreneur is a type of food."}, {"id": 654, "ent": "implant", "catw": "body part", "stmt": "Implant is a type of body part."}, {"id": 655, "ent": "camcorder", "catw": "device", "stmt": "Camcorder is a type of device."}, {"id": 656, "ent": "macadamia", "catw": "living thing", "stmt": "Macadamia is a type of living thing."}, {"id": 657, "ent": "waterfowl", "catw": "game", "stmt": "Waterfowl is a type of game."}, {"id": 658, "ent": "celibate", "catw": "person", "stmt": "Celibate is a type of person."}, {"id": 659, "ent": "rhinoceros", "catw": "person", "stmt": "Rhinoceros is a type of person."}, {"id": 660, "ent": "blazer", "catw": "man-made object", "stmt": "Blazer is a type of man-made object."}, {"id": 661, "ent": "retriever", "catw": "game", "stmt": "Retriever is a type of game."}, {"id": 662, "ent": "antelope", "catw": "footwear", "stmt": "Antelope is a type of footwear."}, {"id": 663, "ent": "wally", "catw": "sport", "stmt": "Wally is a type of sport."}, {"id": 664, "ent": "usurper", "catw": "game", "stmt": "Usurper is a type of game."}, {"id": 665, "ent": "cultivar", "catw": "material", "stmt": "Cultivar is a type of material."}, {"id": 666, "ent": "chalet", "catw": "game", "stmt": "Chalet is a type of game."}, {"id": 667, "ent": "kitchen", "catw": "equipment", "stmt": "Kitchen is a type of equipment."}, {"id": 668, "ent": "matchbox", "catw": "toy", "stmt": "Matchbox is a type of toy."}, {"id": 669, "ent": "homeowner", "catw": "person", "stmt": "Homeowner is a type of person."}, {"id": 670, "ent": "candlestick", "catw": "tool", "stmt": "Candlestick is a type of tool."}, {"id": 671, "ent": "priestess", "catw": "dog", "stmt": "Priestess is a type of dog."}, {"id": 672, "ent": "toddler", "catw": "mammal", "stmt": "Toddler is a type of mammal."}, {"id": 673, "ent": "detonator", "catw": "man-made object", "stmt": "Detonator is a type of man-made object."}, {"id": 674, "ent": "ax", "catw": "man-made object", "stmt": "Ax is a type of man-made object."}, {"id": 675, "ent": "lingerie", "catw": "equipment", "stmt": "Lingerie is a type of equipment."}, {"id": 676, "ent": "biochemist", "catw": "worker", "stmt": "Biochemist is a type of worker."}, {"id": 677, "ent": "clary", "catw": "weapon", "stmt": "Clary is a type of weapon."}, {"id": 678, "ent": "juvenile", "catw": "person", "stmt": "Juvenile is a type of person."}, {"id": 679, "ent": "psychologist", "catw": "worker", "stmt": "Psychologist is a type of worker."}, {"id": 680, "ent": "obstetrician", "catw": "person", "stmt": "Obstetrician is a type of person."}, {"id": 681, "ent": "biologist", "catw": "worker", "stmt": "Biologist is a type of worker."}, {"id": 682, "ent": "camcorder", "catw": "room", "stmt": "Camcorder is a type of room."}, {"id": 683, "ent": "aristocrat", "catw": "fruit", "stmt": "Aristocrat is a type of fruit."}, {"id": 684, "ent": "tripod", "catw": "equipment", "stmt": "Tripod is a type of equipment."}, {"id": 685, "ent": "ashtray", "catw": "metal", "stmt": "Ashtray is a type of metal."}, {"id": 686, "ent": "atheist", "catw": "dog", "stmt": "Atheist is a type of dog."}, {"id": 687, "ent": "airship", "catw": "body part", "stmt": "Airship is a type of body part."}, {"id": 688, "ent": "greenhouse", "catw": "container", "stmt": "Greenhouse is a type of container."}, {"id": 689, "ent": "canoe", "catw": "vehicle", "stmt": "Canoe is a type of vehicle."}, {"id": 690, "ent": "ship", "catw": "tool", "stmt": "Ship is a type of tool."}, {"id": 691, "ent": "jodhpur", "catw": "vehicle", "stmt": "Jodhpur is a type of vehicle."}, {"id": 692, "ent": "coffin", "catw": "footwear", "stmt": "Coffin is a type of footwear."}, {"id": 693, "ent": "wrestler", "catw": "athlete", "stmt": "Wrestler is a type of athlete."}, {"id": 694, "ent": "dominatrix", "catw": "vehicle", "stmt": "Dominatrix is a type of vehicle."}, {"id": 695, "ent": "caribou", "catw": "game", "stmt": "Caribou is a type of game."}, {"id": 696, "ent": "harmonica", "catw": "equipment", "stmt": "Harmonica is a type of equipment."}, {"id": 697, "ent": "genitalia", "catw": "vegetable", "stmt": "Genitalia is a type of vegetable."}, {"id": 698, "ent": "anesthesiologist", "catw": "worker", "stmt": "Anesthesiologist is a type of worker."}, {"id": 699, "ent": "golf", "catw": "equipment", "stmt": "Golf is a type of equipment."}]''')
NEW = json.loads(r'''[{"id": 1000, "ent": "racist", "catw": "plant", "stmt": "Racist is a type of plant."}, {"id": 1001, "ent": "goggles", "catw": "equipment", "stmt": "Goggles is a type of equipment."}, {"id": 1002, "ent": "swatch", "catw": "man-made object", "stmt": "Swatch is a type of man-made object."}, {"id": 1003, "ent": "payer", "catw": "machine", "stmt": "Payer is a type of machine."}, {"id": 1004, "ent": "mater", "catw": "material", "stmt": "Mater is a type of material."}, {"id": 1005, "ent": "fancier", "catw": "game", "stmt": "Fancier is a type of game."}, {"id": 1006, "ent": "electrician", "catw": "plant", "stmt": "Electrician is a type of plant."}, {"id": 1007, "ent": "crucible", "catw": "man-made object", "stmt": "Crucible is a type of man-made object."}, {"id": 1008, "ent": "consul", "catw": "tool", "stmt": "Consul is a type of tool."}, {"id": 1009, "ent": "dropper", "catw": "tool", "stmt": "Dropper is a type of tool."}, {"id": 1010, "ent": "juror", "catw": "material", "stmt": "Juror is a type of material."}, {"id": 1011, "ent": "abbot", "catw": "container", "stmt": "Abbot is a type of container."}, {"id": 1012, "ent": "viper", "catw": "weapon", "stmt": "Viper is a type of weapon."}, {"id": 1013, "ent": "midge", "catw": "living thing", "stmt": "Midge is a type of living thing."}, {"id": 1014, "ent": "godmother", "catw": "bird", "stmt": "Godmother is a type of bird."}, {"id": 1015, "ent": "locator", "catw": "metal", "stmt": "Locator is a type of metal."}, {"id": 1016, "ent": "bassist", "catw": "living thing", "stmt": "Bassist is a type of living thing."}, {"id": 1017, "ent": "immune", "catw": "sport", "stmt": "Immune is a type of sport."}, {"id": 1018, "ent": "fink", "catw": "material", "stmt": "Fink is a type of material."}, {"id": 1019, "ent": "cosmopolitan", "catw": "drink", "stmt": "Cosmopolitan is a type of drink."}, {"id": 1020, "ent": "liza", "catw": "person", "stmt": "Liza is a type of person."}, {"id": 1021, "ent": "stallion", "catw": "mammal", "stmt": "Stallion is a type of mammal."}, {"id": 1022, "ent": "vassal", "catw": "dog", "stmt": "Vassal is a type of dog."}, {"id": 1023, "ent": "dragoon", "catw": "living thing", "stmt": "Dragoon is a type of living thing."}, {"id": 1024, "ent": "traditionalist", "catw": "body part", "stmt": "Traditionalist is a type of body part."}, {"id": 1025, "ent": "stallion", "catw": "living thing", "stmt": "Stallion is a type of living thing."}, {"id": 1026, "ent": "headmaster", "catw": "container", "stmt": "Headmaster is a type of container."}, {"id": 1027, "ent": "tennis", "catw": "container", "stmt": "Tennis is a type of container."}, {"id": 1028, "ent": "godmother", "catw": "material", "stmt": "Godmother is a type of material."}, {"id": 1029, "ent": "magdalen", "catw": "boat", "stmt": "Magdalen is a type of boat."}, {"id": 1030, "ent": "pervert", "catw": "device", "stmt": "Pervert is a type of device."}, {"id": 1031, "ent": "neoliberal", "catw": "tool", "stmt": "Neoliberal is a type of tool."}, {"id": 1032, "ent": "villager", "catw": "living thing", "stmt": "Villager is a type of living thing."}, {"id": 1033, "ent": "salad", "catw": "machine", "stmt": "Salad is a type of machine."}, {"id": 1034, "ent": "supremacist", "catw": "person", "stmt": "Supremacist is a type of person."}, {"id": 1035, "ent": "thermostat", "catw": "metal", "stmt": "Thermostat is a type of metal."}, {"id": 1036, "ent": "guan", "catw": "toy", "stmt": "Guan is a type of toy."}, {"id": 1037, "ent": "bodice", "catw": "clothing", "stmt": "Bodice is a type of clothing."}, {"id": 1038, "ent": "clarinet", "catw": "living thing", "stmt": "Clarinet is a type of living thing."}, {"id": 1039, "ent": "tapeworm", "catw": "vegetable", "stmt": "Tapeworm is a type of vegetable."}, {"id": 1040, "ent": "scold", "catw": "person", "stmt": "Scold is a type of person."}, {"id": 1041, "ent": "ratatouille", "catw": "insect", "stmt": "Ratatouille is a type of insect."}, {"id": 1042, "ent": "panelist", "catw": "metal", "stmt": "Panelist is a type of metal."}, {"id": 1043, "ent": "vegan", "catw": "machine", "stmt": "Vegan is a type of machine."}, {"id": 1044, "ent": "fern", "catw": "metal", "stmt": "Fern is a type of metal."}, {"id": 1045, "ent": "heiress", "catw": "boat", "stmt": "Heiress is a type of boat."}, {"id": 1046, "ent": "mosque", "catw": "container", "stmt": "Mosque is a type of container."}, {"id": 1047, "ent": "literate", "catw": "vegetable", "stmt": "Literate is a type of vegetable."}, {"id": 1048, "ent": "machinist", "catw": "person", "stmt": "Machinist is a type of person."}, {"id": 1049, "ent": "sickle", "catw": "disease", "stmt": "Sickle is a type of disease."}, {"id": 1050, "ent": "heiress", "catw": "fruit", "stmt": "Heiress is a type of fruit."}, {"id": 1051, "ent": "religious", "catw": "material", "stmt": "Religious is a type of material."}, {"id": 1052, "ent": "spelt", "catw": "food", "stmt": "Spelt is a type of food."}, {"id": 1053, "ent": "paella", "catw": "body part", "stmt": "Paella is a type of body part."}, {"id": 1054, "ent": "laser", "catw": "game", "stmt": "Laser is a type of game."}, {"id": 1055, "ent": "tamer", "catw": "dish", "stmt": "Tamer is a type of dish."}, {"id": 1056, "ent": "executor", "catw": "worker", "stmt": "Executor is a type of worker."}, {"id": 1057, "ent": "islander", "catw": "mammal", "stmt": "Islander is a type of mammal."}, {"id": 1058, "ent": "urchin", "catw": "worker", "stmt": "Urchin is a type of worker."}, {"id": 1059, "ent": "obsessive", "catw": "car", "stmt": "Obsessive is a type of car."}, {"id": 1060, "ent": "civilian", "catw": "tree", "stmt": "Civilian is a type of tree."}, {"id": 1061, "ent": "sword", "catw": "man-made object", "stmt": "Sword is a type of man-made object."}, {"id": 1062, "ent": "slingshot", "catw": "musical instrument", "stmt": "Slingshot is a type of musical instrument."}, {"id": 1063, "ent": "cappuccino", "catw": "living thing", "stmt": "Cappuccino is a type of living thing."}, {"id": 1064, "ent": "investor", "catw": "mammal", "stmt": "Investor is a type of mammal."}, {"id": 1065, "ent": "betrothed", "catw": "game", "stmt": "Betrothed is a type of game."}, {"id": 1066, "ent": "condor", "catw": "animal", "stmt": "Condor is a type of animal."}, {"id": 1067, "ent": "magistrate", "catw": "living thing", "stmt": "Magistrate is a type of living thing."}, {"id": 1068, "ent": "turtleneck", "catw": "plant", "stmt": "Turtleneck is a type of plant."}, {"id": 1069, "ent": "mastodon", "catw": "tool", "stmt": "Mastodon is a type of tool."}, {"id": 1070, "ent": "luge", "catw": "sport", "stmt": "Luge is a type of sport."}, {"id": 1071, "ent": "urologist", "catw": "person", "stmt": "Urologist is a type of person."}, {"id": 1072, "ent": "tray", "catw": "dish", "stmt": "Tray is a type of dish."}, {"id": 1073, "ent": "landlord", "catw": "game", "stmt": "Landlord is a type of game."}, {"id": 1074, "ent": "whisperer", "catw": "body part", "stmt": "Whisperer is a type of body part."}, {"id": 1075, "ent": "paella", "catw": "dish", "stmt": "Paella is a type of dish."}, {"id": 1076, "ent": "sandal", "catw": "room", "stmt": "Sandal is a type of room."}, {"id": 1077, "ent": "kicker", "catw": "furniture", "stmt": "Kicker is a type of furniture."}, {"id": 1078, "ent": "metronome", "catw": "device", "stmt": "Metronome is a type of device."}, {"id": 1079, "ent": "portable", "catw": "man-made object", "stmt": "Portable is a type of man-made object."}, {"id": 1080, "ent": "badminton", "catw": "container", "stmt": "Badminton is a type of container."}, {"id": 1081, "ent": "pauper", "catw": "person", "stmt": "Pauper is a type of person."}, {"id": 1082, "ent": "bigot", "catw": "game", "stmt": "Bigot is a type of game."}, {"id": 1083, "ent": "taxpayer", "catw": "vegetable", "stmt": "Taxpayer is a type of vegetable."}, {"id": 1084, "ent": "anthropologist", "catw": "man-made object", "stmt": "Anthropologist is a type of man-made object."}, {"id": 1085, "ent": "farmhouse", "catw": "fruit", "stmt": "Farmhouse is a type of fruit."}, {"id": 1086, "ent": "stockman", "catw": "tree", "stmt": "Stockman is a type of tree."}, {"id": 1087, "ent": "tambourine", "catw": "man-made object", "stmt": "Tambourine is a type of man-made object."}, {"id": 1088, "ent": "outcast", "catw": "person", "stmt": "Outcast is a type of person."}, {"id": 1089, "ent": "contemporary", "catw": "game", "stmt": "Contemporary is a type of game."}, {"id": 1090, "ent": "caliper", "catw": "reptile", "stmt": "Caliper is a type of reptile."}, {"id": 1091, "ent": "sprinkler", "catw": "tool", "stmt": "Sprinkler is a type of tool."}, {"id": 1092, "ent": "screenwriter", "catw": "person", "stmt": "Screenwriter is a type of person."}, {"id": 1093, "ent": "lily", "catw": "reptile", "stmt": "Lily is a type of reptile."}, {"id": 1094, "ent": "kinsman", "catw": "game", "stmt": "Kinsman is a type of game."}, {"id": 1095, "ent": "elitist", "catw": "game", "stmt": "Elitist is a type of game."}, {"id": 1096, "ent": "archivist", "catw": "person", "stmt": "Archivist is a type of person."}, {"id": 1097, "ent": "tadpole", "catw": "building", "stmt": "Tadpole is a type of building."}, {"id": 1098, "ent": "magneto", "catw": "sport", "stmt": "Magneto is a type of sport."}, {"id": 1099, "ent": "catamaran", "catw": "vehicle", "stmt": "Catamaran is a type of vehicle."}, {"id": 1100, "ent": "carburetor", "catw": "animal", "stmt": "Carburetor is a type of animal."}, {"id": 1101, "ent": "illiterate", "catw": "living thing", "stmt": "Illiterate is a type of living thing."}, {"id": 1102, "ent": "transformer", "catw": "sport", "stmt": "Transformer is a type of sport."}, {"id": 1103, "ent": "alarmist", "catw": "person", "stmt": "Alarmist is a type of person."}, {"id": 1104, "ent": "swatch", "catw": "living thing", "stmt": "Swatch is a type of living thing."}, {"id": 1105, "ent": "convalescent", "catw": "flower", "stmt": "Convalescent is a type of flower."}, {"id": 1106, "ent": "guildhall", "catw": "room", "stmt": "Guildhall is a type of room."}, {"id": 1107, "ent": "sprinter", "catw": "tool", "stmt": "Sprinter is a type of tool."}, {"id": 1108, "ent": "pug", "catw": "man-made object", "stmt": "Pug is a type of man-made object."}, {"id": 1109, "ent": "practitioner", "catw": "person", "stmt": "Practitioner is a type of person."}, {"id": 1110, "ent": "sheriff", "catw": "worker", "stmt": "Sheriff is a type of worker."}, {"id": 1111, "ent": "skier", "catw": "sport", "stmt": "Skier is a type of sport."}, {"id": 1112, "ent": "resistor", "catw": "device", "stmt": "Resistor is a type of device."}, {"id": 1113, "ent": "netball", "catw": "sport", "stmt": "Netball is a type of sport."}, {"id": 1114, "ent": "emeritus", "catw": "person", "stmt": "Emeritus is a type of person."}, {"id": 1115, "ent": "spitz", "catw": "boat", "stmt": "Spitz is a type of boat."}, {"id": 1116, "ent": "nettle", "catw": "equipment", "stmt": "Nettle is a type of equipment."}, {"id": 1117, "ent": "artiste", "catw": "body part", "stmt": "Artiste is a type of body part."}, {"id": 1118, "ent": "laptop", "catw": "equipment", "stmt": "Laptop is a type of equipment."}, {"id": 1119, "ent": "cymbal", "catw": "animal", "stmt": "Cymbal is a type of animal."}, {"id": 1120, "ent": "cricketer", "catw": "person", "stmt": "Cricketer is a type of person."}, {"id": 1121, "ent": "hotel", "catw": "room", "stmt": "Hotel is a type of room."}, {"id": 1122, "ent": "compatriot", "catw": "boat", "stmt": "Compatriot is a type of boat."}, {"id": 1123, "ent": "thistle", "catw": "flower", "stmt": "Thistle is a type of flower."}, {"id": 1124, "ent": "colleen", "catw": "clothing", "stmt": "Colleen is a type of clothing."}, {"id": 1125, "ent": "risotto", "catw": "dish", "stmt": "Risotto is a type of dish."}, {"id": 1126, "ent": "pimp", "catw": "person", "stmt": "Pimp is a type of person."}, {"id": 1127, "ent": "mantis", "catw": "animal", "stmt": "Mantis is a type of animal."}, {"id": 1128, "ent": "giraffe", "catw": "fish", "stmt": "Giraffe is a type of fish."}, {"id": 1129, "ent": "armature", "catw": "man-made object", "stmt": "Armature is a type of man-made object."}, {"id": 1130, "ent": "mollie", "catw": "dog", "stmt": "Mollie is a type of dog."}, {"id": 1131, "ent": "azalea", "catw": "fish", "stmt": "Azalea is a type of fish."}, {"id": 1132, "ent": "condom", "catw": "device", "stmt": "Condom is a type of device."}, {"id": 1133, "ent": "mastiff", "catw": "game", "stmt": "Mastiff is a type of game."}, {"id": 1134, "ent": "butch", "catw": "artist", "stmt": "Butch is a type of artist."}, {"id": 1135, "ent": "prefect", "catw": "living thing", "stmt": "Prefect is a type of living thing."}, {"id": 1136, "ent": "breadwinner", "catw": "vegetable", "stmt": "Breadwinner is a type of vegetable."}, {"id": 1137, "ent": "gnu", "catw": "dog", "stmt": "Gnu is a type of dog."}, {"id": 1138, "ent": "supercharger", "catw": "man-made object", "stmt": "Supercharger is a type of man-made object."}, {"id": 1139, "ent": "blaster", "catw": "living thing", "stmt": "Blaster is a type of living thing."}, {"id": 1140, "ent": "serviceman", "catw": "material", "stmt": "Serviceman is a type of material."}, {"id": 1141, "ent": "depressive", "catw": "disease", "stmt": "Depressive is a type of disease."}, {"id": 1142, "ent": "collie", "catw": "animal", "stmt": "Collie is a type of animal."}, {"id": 1143, "ent": "hosiery", "catw": "man-made object", "stmt": "Hosiery is a type of man-made object."}, {"id": 1144, "ent": "buggy", "catw": "container", "stmt": "Buggy is a type of container."}, {"id": 1145, "ent": "tetra", "catw": "mammal", "stmt": "Tetra is a type of mammal."}, {"id": 1146, "ent": "pullover", "catw": "game", "stmt": "Pullover is a type of game."}, {"id": 1147, "ent": "yucca", "catw": "weapon", "stmt": "Yucca is a type of weapon."}, {"id": 1148, "ent": "asthmatic", "catw": "disease", "stmt": "Asthmatic is a type of disease."}, {"id": 1149, "ent": "stockman", "catw": "fabric", "stmt": "Stockman is a type of fabric."}, {"id": 1150, "ent": "scientist", "catw": "person", "stmt": "Scientist is a type of person."}, {"id": 1151, "ent": "acacia", "catw": "tree", "stmt": "Acacia is a type of tree."}, {"id": 1152, "ent": "godmother", "catw": "boat", "stmt": "Godmother is a type of boat."}, {"id": 1153, "ent": "cafeteria", "catw": "building", "stmt": "Cafeteria is a type of building."}, {"id": 1154, "ent": "eclectic", "catw": "furniture", "stmt": "Eclectic is a type of furniture."}, {"id": 1155, "ent": "hibiscus", "catw": "flower", "stmt": "Hibiscus is a type of flower."}, {"id": 1156, "ent": "assemblyman", "catw": "living thing", "stmt": "Assemblyman is a type of living thing."}, {"id": 1157, "ent": "everyman", "catw": "tree", "stmt": "Everyman is a type of tree."}, {"id": 1158, "ent": "sniffer", "catw": "vegetable", "stmt": "Sniffer is a type of vegetable."}, {"id": 1159, "ent": "fiance", "catw": "game", "stmt": "Fiance is a type of game."}, {"id": 1160, "ent": "caddy", "catw": "container", "stmt": "Caddy is a type of container."}, {"id": 1161, "ent": "swami", "catw": "vehicle", "stmt": "Swami is a type of vehicle."}, {"id": 1162, "ent": "reptile", "catw": "animal", "stmt": "Reptile is a type of animal."}, {"id": 1163, "ent": "hourglass", "catw": "food", "stmt": "Hourglass is a type of food."}, {"id": 1164, "ent": "baroness", "catw": "sport", "stmt": "Baroness is a type of sport."}, {"id": 1165, "ent": "boyfriend", "catw": "musical instrument", "stmt": "Boyfriend is a type of musical instrument."}, {"id": 1166, "ent": "omelet", "catw": "weapon", "stmt": "Omelet is a type of weapon."}, {"id": 1167, "ent": "ant", "catw": "reptile", "stmt": "Ant is a type of reptile."}, {"id": 1168, "ent": "breeder", "catw": "room", "stmt": "Breeder is a type of room."}, {"id": 1169, "ent": "lidar", "catw": "footwear", "stmt": "Lidar is a type of footwear."}, {"id": 1170, "ent": "penguin", "catw": "plant", "stmt": "Penguin is a type of plant."}, {"id": 1171, "ent": "batik", "catw": "metal", "stmt": "Batik is a type of metal."}, {"id": 1172, "ent": "pauper", "catw": "living thing", "stmt": "Pauper is a type of living thing."}, {"id": 1173, "ent": "mahatma", "catw": "bird", "stmt": "Mahatma is a type of bird."}, {"id": 1174, "ent": "diocesan", "catw": "person", "stmt": "Diocesan is a type of person."}, {"id": 1175, "ent": "shrub", "catw": "game", "stmt": "Shrub is a type of game."}, {"id": 1176, "ent": "padlock", "catw": "tool", "stmt": "Padlock is a type of tool."}, {"id": 1177, "ent": "astrologer", "catw": "animal", "stmt": "Astrologer is a type of animal."}, {"id": 1178, "ent": "harpsichord", "catw": "equipment", "stmt": "Harpsichord is a type of equipment."}, {"id": 1179, "ent": "mediator", "catw": "living thing", "stmt": "Mediator is a type of living thing."}, {"id": 1180, "ent": "fanatic", "catw": "material", "stmt": "Fanatic is a type of material."}, {"id": 1181, "ent": "tamer", "catw": "sport", "stmt": "Tamer is a type of sport."}, {"id": 1182, "ent": "weevil", "catw": "person", "stmt": "Weevil is a type of person."}, {"id": 1183, "ent": "panelist", "catw": "living thing", "stmt": "Panelist is a type of living thing."}, {"id": 1184, "ent": "hacksaw", "catw": "vegetable", "stmt": "Hacksaw is a type of vegetable."}, {"id": 1185, "ent": "applicant", "catw": "living thing", "stmt": "Applicant is a type of living thing."}, {"id": 1186, "ent": "reactionary", "catw": "boat", "stmt": "Reactionary is a type of boat."}, {"id": 1187, "ent": "tern", "catw": "animal", "stmt": "Tern is a type of animal."}, {"id": 1188, "ent": "registrant", "catw": "man-made object", "stmt": "Registrant is a type of man-made object."}, {"id": 1189, "ent": "marigold", "catw": "person", "stmt": "Marigold is a type of person."}, {"id": 1190, "ent": "barometer", "catw": "device", "stmt": "Barometer is a type of device."}, {"id": 1191, "ent": "paratrooper", "catw": "insect", "stmt": "Paratrooper is a type of insect."}, {"id": 1192, "ent": "chump", "catw": "game", "stmt": "Chump is a type of game."}, {"id": 1193, "ent": "vertebrate", "catw": "mammal", "stmt": "Vertebrate is a type of mammal."}, {"id": 1194, "ent": "slingshot", "catw": "fish", "stmt": "Slingshot is a type of fish."}, {"id": 1195, "ent": "walrus", "catw": "fruit", "stmt": "Walrus is a type of fruit."}, {"id": 1196, "ent": "workaholic", "catw": "worker", "stmt": "Workaholic is a type of worker."}, {"id": 1197, "ent": "guesthouse", "catw": "metal", "stmt": "Guesthouse is a type of metal."}, {"id": 1198, "ent": "housemate", "catw": "musical instrument", "stmt": "Housemate is a type of musical instrument."}, {"id": 1199, "ent": "groupie", "catw": "person", "stmt": "Groupie is a type of person."}, {"id": 1200, "ent": "abalone", "catw": "food", "stmt": "Abalone is a type of food."}, {"id": 1201, "ent": "seller", "catw": "artist", "stmt": "Seller is a type of artist."}, {"id": 1202, "ent": "grenade", "catw": "man-made object", "stmt": "Grenade is a type of man-made object."}, {"id": 1203, "ent": "zebra", "catw": "vegetable", "stmt": "Zebra is a type of vegetable."}, {"id": 1204, "ent": "serviceman", "catw": "tree", "stmt": "Serviceman is a type of tree."}, {"id": 1205, "ent": "colonel", "catw": "living thing", "stmt": "Colonel is a type of living thing."}, {"id": 1206, "ent": "colonial", "catw": "metal", "stmt": "Colonial is a type of metal."}, {"id": 1207, "ent": "brew", "catw": "disease", "stmt": "Brew is a type of disease."}, {"id": 1208, "ent": "guan", "catw": "bird", "stmt": "Guan is a type of bird."}, {"id": 1209, "ent": "skiff", "catw": "container", "stmt": "Skiff is a type of container."}, {"id": 1210, "ent": "gander", "catw": "living thing", "stmt": "Gander is a type of living thing."}, {"id": 1211, "ent": "cockatoo", "catw": "reptile", "stmt": "Cockatoo is a type of reptile."}, {"id": 1212, "ent": "shaman", "catw": "living thing", "stmt": "Shaman is a type of living thing."}, {"id": 1213, "ent": "businesswoman", "catw": "worker", "stmt": "Businesswoman is a type of worker."}, {"id": 1214, "ent": "nomad", "catw": "worker", "stmt": "Nomad is a type of worker."}, {"id": 1215, "ent": "armature", "catw": "equipment", "stmt": "Armature is a type of equipment."}, {"id": 1216, "ent": "turbine", "catw": "man-made object", "stmt": "Turbine is a type of man-made object."}, {"id": 1217, "ent": "scythe", "catw": "equipment", "stmt": "Scythe is a type of equipment."}, {"id": 1218, "ent": "ale", "catw": "drink", "stmt": "Ale is a type of drink."}, {"id": 1219, "ent": "cappuccino", "catw": "person", "stmt": "Cappuccino is a type of person."}, {"id": 1220, "ent": "vine", "catw": "disease", "stmt": "Vine is a type of disease."}, {"id": 1221, "ent": "bookshelf", "catw": "furniture", "stmt": "Bookshelf is a type of furniture."}, {"id": 1222, "ent": "dirk", "catw": "worker", "stmt": "Dirk is a type of worker."}, {"id": 1223, "ent": "essayist", "catw": "material", "stmt": "Essayist is a type of material."}, {"id": 1224, "ent": "clarion", "catw": "metal", "stmt": "Clarion is a type of metal."}, {"id": 1225, "ent": "boudoir", "catw": "game", "stmt": "Boudoir is a type of game."}, {"id": 1226, "ent": "adjuster", "catw": "flower", "stmt": "Adjuster is a type of flower."}, {"id": 1227, "ent": "airbrush", "catw": "building", "stmt": "Airbrush is a type of building."}, {"id": 1228, "ent": "schmuck", "catw": "material", "stmt": "Schmuck is a type of material."}, {"id": 1229, "ent": "mystic", "catw": "room", "stmt": "Mystic is a type of room."}, {"id": 1230, "ent": "transponder", "catw": "equipment", "stmt": "Transponder is a type of equipment."}, {"id": 1231, "ent": "purist", "catw": "drink", "stmt": "Purist is a type of drink."}, {"id": 1232, "ent": "rani", "catw": "mammal", "stmt": "Rani is a type of mammal."}, {"id": 1233, "ent": "overseer", "catw": "living thing", "stmt": "Overseer is a type of living thing."}, {"id": 1234, "ent": "locator", "catw": "tool", "stmt": "Locator is a type of tool."}, {"id": 1235, "ent": "pinto", "catw": "flower", "stmt": "Pinto is a type of flower."}, {"id": 1236, "ent": "reeve", "catw": "bird", "stmt": "Reeve is a type of bird."}, {"id": 1237, "ent": "valet", "catw": "device", "stmt": "Valet is a type of device."}, {"id": 1238, "ent": "evergreen", "catw": "plant", "stmt": "Evergreen is a type of plant."}, {"id": 1239, "ent": "fisherman", "catw": "game", "stmt": "Fisherman is a type of game."}, {"id": 1240, "ent": "chipmunk", "catw": "mammal", "stmt": "Chipmunk is a type of mammal."}, {"id": 1241, "ent": "magdalen", "catw": "person", "stmt": "Magdalen is a type of person."}, {"id": 1242, "ent": "sahib", "catw": "vegetable", "stmt": "Sahib is a type of vegetable."}, {"id": 1243, "ent": "marauder", "catw": "boat", "stmt": "Marauder is a type of boat."}, {"id": 1244, "ent": "duce", "catw": "living thing", "stmt": "Duce is a type of living thing."}, {"id": 1245, "ent": "bobby", "catw": "animal", "stmt": "Bobby is a type of animal."}, {"id": 1246, "ent": "scapegoat", "catw": "bird", "stmt": "Scapegoat is a type of bird."}, {"id": 1247, "ent": "sinner", "catw": "game", "stmt": "Sinner is a type of game."}, {"id": 1248, "ent": "mastodon", "catw": "flower", "stmt": "Mastodon is a type of flower."}, {"id": 1249, "ent": "breeder", "catw": "person", "stmt": "Breeder is a type of person."}, {"id": 1250, "ent": "bathtub", "catw": "equipment", "stmt": "Bathtub is a type of equipment."}, {"id": 1251, "ent": "infielder", "catw": "food", "stmt": "Infielder is a type of food."}, {"id": 1252, "ent": "mustang", "catw": "living thing", "stmt": "Mustang is a type of living thing."}, {"id": 1253, "ent": "surfer", "catw": "athlete", "stmt": "Surfer is a type of athlete."}, {"id": 1254, "ent": "converter", "catw": "equipment", "stmt": "Converter is a type of equipment."}, {"id": 1255, "ent": "prior", "catw": "living thing", "stmt": "Prior is a type of living thing."}, {"id": 1256, "ent": "prowler", "catw": "animal", "stmt": "Prowler is a type of animal."}, {"id": 1257, "ent": "juror", "catw": "worker", "stmt": "Juror is a type of worker."}, {"id": 1258, "ent": "dermatologist", "catw": "material", "stmt": "Dermatologist is a type of material."}, {"id": 1259, "ent": "policeman", "catw": "food", "stmt": "Policeman is a type of food."}, {"id": 1260, "ent": "triceps", "catw": "weapon", "stmt": "Triceps is a type of weapon."}, {"id": 1261, "ent": "tun", "catw": "boat", "stmt": "Tun is a type of boat."}, {"id": 1262, "ent": "speedometer", "catw": "equipment", "stmt": "Speedometer is a type of equipment."}, {"id": 1263, "ent": "beggar", "catw": "sport", "stmt": "Beggar is a type of sport."}, {"id": 1264, "ent": "workaholic", "catw": "bird", "stmt": "Workaholic is a type of bird."}, {"id": 1265, "ent": "lathe", "catw": "device", "stmt": "Lathe is a type of device."}, {"id": 1266, "ent": "gland", "catw": "organ", "stmt": "Gland is a type of organ."}, {"id": 1267, "ent": "starling", "catw": "vegetable", "stmt": "Starling is a type of vegetable."}, {"id": 1268, "ent": "ladle", "catw": "tool", "stmt": "Ladle is a type of tool."}, {"id": 1269, "ent": "pontifical", "catw": "insect", "stmt": "Pontifical is a type of insect."}, {"id": 1270, "ent": "collie", "catw": "vegetable", "stmt": "Collie is a type of vegetable."}, {"id": 1271, "ent": "dramatist", "catw": "equipment", "stmt": "Dramatist is a type of equipment."}, {"id": 1272, "ent": "goldfish", "catw": "animal", "stmt": "Goldfish is a type of animal."}, {"id": 1273, "ent": "corporal", "catw": "weapon", "stmt": "Corporal is a type of weapon."}, {"id": 1274, "ent": "overseer", "catw": "worker", "stmt": "Overseer is a type of worker."}, {"id": 1275, "ent": "shill", "catw": "artist", "stmt": "Shill is a type of artist."}, {"id": 1276, "ent": "prisoner", "catw": "game", "stmt": "Prisoner is a type of game."}, {"id": 1277, "ent": "contemporary", "catw": "artist", "stmt": "Contemporary is a type of artist."}, {"id": 1278, "ent": "gorilla", "catw": "athlete", "stmt": "Gorilla is a type of athlete."}, {"id": 1279, "ent": "cursor", "catw": "tool", "stmt": "Cursor is a type of tool."}, {"id": 1280, "ent": "socialite", "catw": "living thing", "stmt": "Socialite is a type of living thing."}, {"id": 1281, "ent": "milo", "catw": "living thing", "stmt": "Milo is a type of living thing."}, {"id": 1282, "ent": "techie", "catw": "person", "stmt": "Techie is a type of person."}, {"id": 1283, "ent": "madder", "catw": "living thing", "stmt": "Madder is a type of living thing."}, {"id": 1284, "ent": "troublemaker", "catw": "vegetable", "stmt": "Troublemaker is a type of vegetable."}, {"id": 1285, "ent": "bricklayer", "catw": "athlete", "stmt": "Bricklayer is a type of athlete."}, {"id": 1286, "ent": "guitar", "catw": "musical instrument", "stmt": "Guitar is a type of musical instrument."}, {"id": 1287, "ent": "fizz", "catw": "game", "stmt": "Fizz is a type of game."}, {"id": 1288, "ent": "pantry", "catw": "room", "stmt": "Pantry is a type of room."}, {"id": 1289, "ent": "bisexual", "catw": "living thing", "stmt": "Bisexual is a type of living thing."}, {"id": 1290, "ent": "jaguar", "catw": "animal", "stmt": "Jaguar is a type of animal."}, {"id": 1291, "ent": "layman", "catw": "plant", "stmt": "Layman is a type of plant."}, {"id": 1292, "ent": "namesake", "catw": "sport", "stmt": "Namesake is a type of sport."}, {"id": 1293, "ent": "moped", "catw": "container", "stmt": "Moped is a type of container."}, {"id": 1294, "ent": "megaphone", "catw": "food", "stmt": "Megaphone is a type of food."}, {"id": 1295, "ent": "skeptic", "catw": "boat", "stmt": "Skeptic is a type of boat."}, {"id": 1296, "ent": "novelist", "catw": "artist", "stmt": "Novelist is a type of artist."}, {"id": 1297, "ent": "condom", "catw": "equipment", "stmt": "Condom is a type of equipment."}, {"id": 1298, "ent": "yacht", "catw": "dog", "stmt": "Yacht is a type of dog."}, {"id": 1299, "ent": "vine", "catw": "plant", "stmt": "Vine is a type of plant."}]''')
ITEMS = OLD + NEW
r12 = json.load(open(E112))
L = r12['L']
CHOICE = {(w, k): v for w, k, v in r12['choice']}
ST = {(w, k): tuple(v) for w, k, *v in r12['steps']}
AR = {(w, k, C): tuple(v) for w, k, C, *v in r12['arrive']}
# 確かめ：資料を足さない（srcs=()）と、E11-2 の6段の鎖と同じ
CH12 = {i: nd for i, nd, lk in r12['chains']}
bad = sum({(w, k) for (t, w, k) in follow(L, CHOICE, {}, it['ent'], 6, ())[0]} != {(w, k) for w, k, d in CH12[it['id']]} for it in ITEMS)
assert bad == 0, f'資料なしの鎖が E11-2 と {bad} 問で違う'
with gzip.open(WIKT, 'rt', encoding='utf-8') as f: WKD = json.load(f)['n']
print(len(ITEMS), '問  語', len(L), '  資料なしの鎖は E11-2 と一致  辞書の見出し', len(WKD))

## 1. NLI と Gemma を読み込む（GPU、2〜4分）

In [ ]:
import torch, spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
nlp = spacy.load("en_core_web_sm", disable=["ner"])
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["neutral"], lab["contradiction"]]
tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_DEFS_MID + 1)]
YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]

def _nli(P, H, B=64):
    out = []
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER].cpu().numpy()
        out += [(round(float(p[0]), 4), round(float(p[2]), 4)) for p in pr]
    return out
def nli(trip):
    # (w, k, c)：前提「"w" means: 説明文」、仮説「W is a type of c.」（E8n と同じ）
    return _nli([premise(w, L[w][k]["def"]) for w, k, c in trip], [hyp(w, c) for w, k, c in trip])
def nli_sent(trip):
    # (文, w, c)：前提＝文そのもの、仮説「W is a type of c.」
    return _nli([s for s, w, c in trip], [hyp(w, c) for s, w, c in trip])

def _pick(qs, ns, B=16):
    out = []
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for row, n in zip(last, ns[i:i + B]):
            out.append(int(torch.softmax(row[NUM[:n]], -1).argmax()))
    return out
def gemma_choice(jobs):
    # (w, k) → 頭の名詞 g の意味の番号。聞き方は E8n の方法D と同じ
    qs = [choice_prompt(w, L[w][k]["def"], L[w][k]["genus"], [x["def"] for x in L[L[w][k]["genus"]]]) for w, k in jobs]
    return _pick(qs, [len(L[L[w][k]["genus"]]) for w, k in jobs])
def gemma_sense(jobs):
    # (文, 語, [説明文]) → 文の中のその語の意味の番号
    return _pick([sense_in_sentence_prompt(s, w, ds) for s, w, ds in jobs], [len(ds) for s, w, ds in jobs])
SELF = {}
def self_write(words, B=32):
    todo = [w for w in words if w not in SELF]
    for i in range(0, len(todo), B):
        bt = todo[i:i + B]
        ps = [tok.apply_chat_template([{"role": "user", "content": self_prompt(w)}], tokenize=False, add_generation_prompt=True) for w in bt]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            g = gm.generate(**enc, max_new_tokens=SELF_TOKENS, do_sample=False)
        for w, row in zip(bt, g[:, enc["input_ids"].shape[1]:]):
            SELF[w] = {"raw": tok.decode(row, skip_special_tokens=True), "sent": None}
            SELF[w]["sent"] = clean_self(SELF[w]["raw"])
    return {w: SELF[w]["sent"] for w in words}
def yes_no(qs, B=32):
    out = []
    for i in range(0, len(qs), B):
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs[i:i + B]]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            r = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        out.extend((r[:, YES] - r[:, NO]).cpu().numpy().round(3).tolist())
    return out
# E14-1：継ぎ足しの文の判定を Gemma に（聞き方は1つだけ、PLAN_E14-1.md）
def says_prompt(s, w, c):
    return f"Sentence: {s}\n\nDoes this sentence say that a \"{w}\" is a kind of \"{c}\"? Answer Yes or No only."
def gemma_says(trip, B=32):
    d = yes_no([says_prompt(s, w, c) for s, w, c in trip], B)
    return [(round(float(1 / (1 + np.exp(-x))), 4), 0.0) for x in d]
nli_sent = gemma_says
gkb_sents = lambda words: {}      # E14-2 では使わない
self_write = lambda words: {}     # E14-2 では使わない
print(says_prompt("Granddaughters trump even a four-homer win any time.", "granddaughter", "win"))
print(gemma_says([("Granddaughters trump even a four-homer win any time.", "granddaughter", "win"),
                  ("Hominids are primates with no tails.", "hominid", "primate")]))
# E14-2：Wikipedia の記事の最初の1文（公開の要約 API。曖昧さ回避など standard でないページは使わない）
import requests
UA = {"User-Agent": "AKANE-research-colab/1.1 (non-commercial research; low request rate)"}
API = "https://en.wikipedia.org/w/api.php"
WIKI = load_gz("E142v2_wiki") or {}
N_REQ, N_429 = [0], [0]
def _wiki_batch(words):
    # v2：1回の問い合わせで20語まで（記事の冒頭の説明だけ、平文）。1本ずつ・1秒あける。断られたら言われた時間だけ待つ。
    # 曖昧さ回避のページ（pageprops に disambiguation）と記事のない語は None。転送（hominid→Hominidae）はたどる。
    titles = {w: w[:1].upper() + w[1:] for w in words}
    params = {"action": "query", "format": "json", "formatversion": "2", "prop": "extracts|pageprops",
              "exintro": "1", "explaintext": "1", "exlimit": "20", "ppprop": "disambiguation", "redirects": "1",
              "maxlag": "5", "titles": "|".join(titles.values())}
    for i in range(8):
        time.sleep(1.0)
        try:
            N_REQ[0] += 1
            r = requests.get(API, params=params, headers=UA, timeout=30)
            if r.status_code == 429 or r.status_code >= 500 or "maxlag" in r.text[:200]:
                N_429[0] += 1
                time.sleep(float(r.headers.get("Retry-After", 0) or 0) or min(60, 5 * 2 ** i))
                continue
            if r.status_code != 200:
                time.sleep(min(60, 5 * 2 ** i)); continue
            q = r.json().get("query", {})
            go = {}
            for x in q.get("normalized", []): go[x["from"]] = x["to"]
            red = {x["from"]: x["to"] for x in q.get("redirects", [])}
            pages = {p["title"]: p for p in q.get("pages", [])}
            out = {}
            for w, t in titles.items():
                t = go.get(t, t); t = red.get(t, t)
                p = pages.get(t)
                if not p or p.get("missing") or p.get("invalid") or "disambiguation" in (p.get("pageprops") or {}):
                    out[w] = None
                else:
                    out[w] = {"title": p["title"], "extract": p.get("extract") or ""}
            return out
        except Exception:
            time.sleep(min(60, 5 * 2 ** i))
    return {w: {"error": True} for w in words}
def wiki_sents(words):
    todo = [w for w in words if w not in WIKI or (WIKI[w] or {}).get("error")]
    if todo:
        t = time.time()
        for s in range(0, len(todo), 20):
            WIKI.update(_wiki_batch(todo[s:s + 20]))
        save_gz(WIKI, "E142v2_wiki")
        print(f"  Wikipedia {len(todo)}語 {time.time() - t:.0f}秒  取れた {sum(1 for w in todo if WIKI[w] and WIKI[w].get('extract'))}  "
              f"記事なし・曖昧さ回避 {sum(1 for w in todo if WIKI[w] is None)}  失敗 {sum(1 for w in todo if WIKI[w] and WIKI[w].get('error'))}  "
              f"問い合わせ {N_REQ[0]}  待たされた {N_429[0]}")
        bad = sum(1 for w in todo if WIKI[w] and WIKI[w].get('error'))
        assert bad <= 0.05 * len(todo), "Wikipedia に断られ続けています。少し時間をおいて、もう一度「すべてのセルを実行」してください（取れた所は残ります）"
    out = {}
    for w in words:
        r = WIKI.get(w)
        if r and r.get("extract"):
            sents = list(nlp(r["extract"][:600]).sents)
            if sents:
                out[w] = sents[0].text.strip()
    return out
print(wiki_sents(["zoologist", "hominid", "kitten", "ant", "artist", "bathtub", "mercury"]))
print("読み込み済み")

## 3. 継ぎ足す（GPU、40〜60分）。終わったら E142v2_grow.json.gz がドライブにできます

In [ ]:
saved = load_gz("E142v2_grow")
if saved is not None:
    L, GR, SELF, STAT = saved["L"], saved["GR"], saved["self"], saved["stat"]
    CHOICE = {(w, k): v for w, k, v in saved["choice"]}
    ST = {(w, k): tuple(v) for w, k, *v in saved["steps"]}
    AR = {(w, k, C): tuple(v) for w, k, C, *v in saved["arrive"]}
    ARG = {(w, i, C): tuple(v) for w, i, C, *v in saved["arrive_g"]}
    print("ドライブから読み込み:", STAT)
else:
    GR, ARG = {}, {}
    t = time.time()
    STAT = grow(ITEMS, L, CHOICE, ST, AR, GR, ARG, WKD, nlp, nli, nli_sent, gemma_choice, gemma_sense, gkb_sents, self_write, wiki_sents=wiki_sents)
    print("足した物:", STAT, f"{time.time() - t:.0f}秒")
    save_gz({"L": L, "GR": GR, "self": SELF, "stat": STAT,
             "choice": [[w, k, v] for (w, k), v in CHOICE.items()],
             "steps": [[w, k, *v] for (w, k), v in ST.items()],
             "arrive": [[w, k, C, *v] for (w, k, C), v in AR.items()],
             "arrive_g": [[w, i, C, *v] for (w, i, C), v in ARG.items()]}, "E142v2_grow")
CHAINS = chains(ITEMS, L, CHOICE, GR)
print("点の種類と段:", dict(sorted(collections.Counter((t, d) for nd, lk in CHAINS.values() for t, w, k, d in nd).items())))
ex = [w for w in ("zoologist", "human", "person", "granddaughter") if GR.get(w)]
for w in ex: print(w, [(g["src"], g["sent"], g["parent"], g["step"][0]) for g in GR[w]])